In [1]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline
import seaborn as sns
from src.selection_visualization import plot_lasso_vs_et

from src.split_visualization import plot_dst_split_distribution

from src.estimators import NaivePersistence, DirectARBaseline, XGBoostDst, LightGBMDst
from src.evaluate         import compute_metrics
from src.runner           import fit_model, log_metrics_run, print_horizon_metrics, run_segment
from src.mlflow_tracking  import setup_mlflow
from src.config import STORM_THR, K_HORIZONS
from src.splits import BOUNDARIES, PURGE_H, build_masks, segment_mask
from src.features import build_feature_sets
from src.config import STORM_THR, K_HORIZONS
from src.features import build_feature_sets

from scipy import stats


In [2]:
setup_mlflow()

<Experiment: artifact_location='file:///C:/Temp/python-start/work/PracticalProjects/cosmic-ray-storm/notebooks/../mlruns/292621262165687921', creation_time=1782395978156, effective_trace_archival_retention=None, experiment_id='292621262165687921', last_update_time=1782395978156, lifecycle_stage='active', name='cosmic_ray_storm_prediction', tags={}, trace_location=None, workspace='default'>

# Cosmic Ray Storm Prediction — Machine Learning

The Feature Engineering stage (`cosmic_ray_storm_prediction_FE.ipynb`) constructed the supervised learning dataset and defined the experimental protocol. The present notebook implements the remaining stages of the machine learning workflow.

## Introduction

The workflow consists of the following stages:

- **Feature Screening** — identifies the predictor subset used consistently throughout all subsequent experiments.
- **Scientific Validation** — establishes forecasting baselines, evaluates the predictive contribution of neutron-monitor observations, and determines the operational forecasting horizon.
- **Operational Forecasting Model** — implements the conclusions of the scientific validation to construct and evaluate the final forecasting model.

All model training and evaluation follow the chronological train–validation–test protocol defined in the Feature Engineering stage.

## Feature Screening and Selection

This stage supports the **forecasting task** defined in the Problem Formulation by identifying the predictor subset used consistently throughout all subsequent modelling experiments. Using a common feature set ensures that model comparisons reflect differences in forecasting methodology rather than differences in the available predictors.

The engineered feature matrix contains 33 candidate predictors. Before model development, a dedicated screening stage removes redundant variables while retaining those that consistently contribute to forecasting performance.

To keep the modelling workflow reproducible, feature screening is implemented in the helper notebook `feature_selection.ipynb`. The resulting predictor subset is then reused unchanged during both the Scientific Validation experiments and the Operational Forecasting Model.

Two complementary selection methods are combined: LASSO, which performs embedded linear feature selection, and ExtraTrees, which estimates non-linear feature importance. The final predictor subset is obtained by combining the selections across methods and forecasting horizons.

Using the strict selection criterion (`min_votes = 2`), 23 predictors are retained by both methods. Together with the physically motivated inclusion of `d_neutron`, this produces the final set of 24 predictors used throughout the remainder of the study.

### Feature Scaling

The predictors span different physical units and numerical ranges, including nanoteslas, kilometres per second, kelvin, and neutron counts. Before feature screening, the predictor matrix is standardised using `StandardScaler` so that all variables are expressed on a comparable scale.

Each feature is transformed to zero mean and unit variance:

$$
\tilde{x}_{t,j}=\frac{x_{t,j}-\mu_j}{\sigma_j},
$$

where $\mu_j$ and $\sigma_j$ are the mean and standard deviation of feature $j$ estimated from the training segment only. The fitted transformation is then applied unchanged to the validation and test segments, preventing distribution leakage.

Standardisation is performed only for the feature screening stage. The scaler is fitted using the training data and reused for all subsequent segments, ensuring that no information from future observations influences feature selection.

### Feature Screening

Feature screening is implemented in the helper notebook `feature_selection.ipynb`. Separating this stage from the main workflow avoids repeating the computationally intensive selection procedure while ensuring that all subsequent experiments use the same predictor subset.

The helper notebook loads the feature matrix and shared experimental configuration produced during Feature Engineering:

- `feat_split.parquet` — imputed, unscaled feature matrix;
- `split_masks.pkl` — train, validation, and test segment masks;
- `context_constants.pkl` — shared project constants (`FEATURE_COLS`, `K_HORIZONS`, `STORM_THR`, `PURGE_H`, and `BOUNDARIES`).

### Methodology

Feature selection combines two complementary methods implemented in the custom `FeatureSelectionPipeline`. The methods were chosen for three reasons: they capture both linear and non-linear predictor relationships, support sample weighting, and are compatible with the chronological training procedure adopted in this study.

- **LASSO** performs embedded linear feature selection using `LassoCV` with `TimeSeriesSplit (n_splits = 5)`. The $\ell_1$ penalty shrinks irrelevant coefficients exactly to zero, producing a compact predictor set while selecting the regularisation strength by cross-validation.

- **ExtraTrees** estimates feature importance using an ensemble of 100 extremely randomised trees. Unlike LASSO, it captures non-linear relationships and interactions between predictors without assuming linear dependence on $D_{st}$.

Both methods support `sample_weight`, allowing storm observations ($D_{st} < -50$ nT) to receive higher weight during feature selection. This is essential because storm periods are strongly underrepresented in the chronological training set. A predictor is retained only if it is selected by both methods for at least one forecast horizon (`min_votes = 2`).

Using the strict voting criterion (`min_votes = 2`), 23 predictors were selected consistently by both screening methods. The final modelling subset contains 24 predictors, with `d_neutron` retained as a physics-guided feature despite receiving a single vote. This preserves the derived neutron precursor signal while maintaining a predominantly data-driven selection procedure.

### Results

> **Feature Screening Results:**
> - The strict voting criterion (`min_votes = 2`) retained 23 predictors. The final modelling subset contains 24 predictors, with `d_neutron` retained as a physics-guided feature despite receiving a single vote. Only `dbz_dt` was rejected by both selection methods at all five forecast horizons.
>
> - **LASSO** selected between 17 and 22 predictors depending on the forecast horizon. The regularisation parameter was highest at the 1-hour horizon (α = 1.641), decreased towards the 12-hour horizon (α = 0.199), and increased again at 21 hours (α = 0.456), indicating that the amount of regularisation required varies with prediction horizon.
>
> - **ExtraTrees** retained 17 predictors at every forecast horizon. The median importance threshold therefore produced a stable feature subset across all prediction horizons.
>
> - **`d_neutron`** was selected only by the storm-weighted LASSO model (7-hour horizon) and not by ExtraTrees. Because it represents the temporal gradient of the neutron flux—a physically motivated Forbush Decrease precursor—it is retained in the final modelling subset [KIS25].
>
> - The final predictor set preserves all major physical components of the Sun–Earth system: interplanetary magnetic field, solar wind plasma, solar activity, neutron monitor observations, temporal history, and solar cycle phase.

In [3]:
# ── Feature Screening Checkpoint Load ────────────────────────────────────
# Loads feature selection results from feature_selection.ipynb.
# Project constants are imported from src/config.py.


feat_data         = pd.read_parquet('data/processed/feat_split.parquet')
fs_results        = joblib.load('models/feature_selection_results.pkl')
SELECTED_FEATURES = fs_results['selected_final']
scaler            = fs_results['scaler']
feature_sets      = build_feature_sets(SELECTED_FEATURES)

print(f"SELECTED_FEATURES : {len(SELECTED_FEATURES)} features")
print(f"K_HORIZONS        : {K_HORIZONS}")
print(f"STORM_THR         : {STORM_THR} nT")
print(f"Scaler            : {scaler}")

SELECTED_FEATURES : 24 features
K_HORIZONS        : [1, 3, 7, 12, 21]
STORM_THR         : -50.0 nT
Scaler            : StandardScaler()


In [4]:
# ── Vote Counts Summary ───────────────────────────────────────────────────
vote_counts = fs_results['vote_counts_raw']

print(f"\n{'Feature':>30} {'Max votes':>10} {'Selected':>10}")
print("─" * 55)
for feature, votes in vote_counts.sort_values(ascending=False).items():
    selected = "+" if feature in SELECTED_FEATURES else "-"
    print(f"{feature:>30} {int(votes):>10} {selected:>10}")


                       Feature  Max votes   Selected
───────────────────────────────────────────────────────
                        bz_gsm          2          +
                      sw_speed          2          +
                    sw_density          2          +
                   sw_pressure          2          +
                       e_field          2          +
                          f107          2          +
                   mach_alfven          2          +
                neutron_counts          2          +
                           ssn          2          +
                    bz_acc_12h          2          +
                   bz_gsm_lag1          2          +
                     bz_acc_6h          2          +
                     bz_acc_3h          2          +
                 sw_speed_lag1          2          +
                  bz_gsm_lag21          2          +
                  bz_gsm_lag12          2          +
                   bz_gsm_lag3          2 

### Physical interpretation of excluded features

Features excluded from the final modelling subset fall into two categories. The first consists of `dbz_dt`, which was rejected by both selection methods at every forecast horizon (`votes = 0`). The second comprises nine predictors receiving a single vote, indicating that they were supported by only one selection method and therefore lacked cross-method agreement.

`bz_gsm_lag7` is the southward IMF component measured 7 hours before the current observation — precisely at the Burton ring current decay timescale (τ ≈ 7–8 hours). The ring current acts as an integrator: the magnetic field state 7 hours ago is already encoded in the present $D_{st}$ value through exponential decay. Short lags (`bz_gsm_lag1`, `bz_gsm_lag3`) capture the onset of southward turning; the accumulated Bz features capture the injection history over 3, 6, and 12 hours. The absence of cross-method agreement suggests that its predictive information is already represented by the neighbouring lag and accumulated IMF features.

`sw_speed_lag21` is the solar wind bulk velocity measured 21 hours before the current observation. Long-term baseline solar wind conditions are better represented by `ssn` and `f107` already in the feature set, while the active CME driver is captured by shorter lags. No additional predictive information is expected beyond that already captured by the shorter lags and the long-term solar activity indicators.

`neutron_counts_lag1` is the LMKS neutron monitor count rate 1 hour before the current observation. The Forbush Decrease unfolds over 6–24 hours — a 1-hour lag is physically indistinguishable from the current value at hourly resolution. The signal is already captured by `neutron_counts` and more cleanly by `d_neutron`.

`neutron_counts_lag21` is the neutron monitor count rate at the outer boundary of the 7–21h Forbush Decrease lead time window [KIS25]. At this lag the precursor signal is already attenuating and does not add detectable information beyond the shorter retained lags.

`sw_temp` and `plasma_beta` Both variables were retained by LASSO at some horizons but consistently fell below the ExtraTrees importance threshold. Their contribution appears to overlap with the retained solar wind state variables (`sw_speed`, `sw_density`, and `sw_pressure`).

`dbz_dt` represents the hour-to-hour change in the southward IMF component. At hourly resolution it provides no consistent predictive contribution beyond the retained accumulated IMF features (`bz_acc_3h`, `bz_acc_6h`, `bz_acc_12h`), which better represent the sustained southward IMF driving ring current injection in the Burton model [BUR75]. Accordingly, `dbz_dt` was rejected by both selection methods at every forecast horizon.

## Validation of Neutron Flux Predictive Gain

This stage addresses the **research objective** and **forecasting horizons** defined in the Problem Formulation. Its purpose is to determine whether neutron-monitor observations provide predictive information beyond that already contained in the conventional OMNI solar wind measurements and to identify the most suitable operational forecasting horizon.

Rather than directly constructing the final forecasting model, the analysis proceeds through a sequence of controlled experiments. The temporal split, learning algorithm, training procedure, and evaluation protocol remain fixed throughout, while the available predictor set is progressively expanded. This controlled design ensures that any observed change in predictive performance can be attributed to the additional predictors rather than to differences in model configuration.

The validation consists of two consecutive stages:

1. **Establishing Predictability.** Quantify the predictive skill obtainable from persistence, the current geomagnetic state, and conventional OMNI solar wind observations, establishing the reference baselines for all subsequent comparisons.

2. **Neutron Flux Predictive Gain.** Progressively augment the OMNI baseline with neutron-derived predictors, quantify the incremental predictive value of each neutron representation, and identify the forecasting horizon at which neutron-derived information provides the greatest relative contribution.

The conclusions of this section determine whether neutron observations provide measurable predictive value and identify the forecast horizon and neutron representation carried forward to the Operational Forecasting Model.

**Evaluation metrics.** Model performance is assessed using four complementary metrics. **RMSE** measures the overall prediction error, while **Storm RMSE** evaluates accuracy only during geomagnetic storms ($D_{st}<-50$ nT), the primary region of interest. **MASE** expresses forecast skill relative to the Naive Persistence baseline, with values below 1 indicating an improvement over persistence. Finally, the **Diebold–Mariano (DM) test** assesses whether the difference in forecast accuracy between two competing models is statistically significant.

All models are evaluated using the same four metrics.

**Root Mean Squared Error (RMSE)** measures the overall prediction error:

$$
\mathrm{RMSE}=
\sqrt{\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2},
$$

where $y_i$ is the observed $D_{st}$, $\hat{y}_i$ is the predicted value, and $N$ is the number of observations. Lower values indicate better overall accuracy.

**Storm RMSE** is computed using only storm observations ($D_{st}<-50$ nT):

$$
\mathrm{Storm\ RMSE}=
\sqrt{\frac{1}{N_{\rm storm}}
\sum_{D_{st}<-50}(y_i-\hat{y}_i)^2},
$$

where $N_{\rm storm}$ is the number of storm samples. This metric evaluates model performance during geomagnetic storms.

**Mean Absolute Scaled Error (MASE)** compares the prediction error with the average one-step error in the training series:

$$
\mathrm{MASE}=
\frac{\frac{1}{N}\sum |y_i-\hat{y}_i|}
{\frac{1}{N_{\rm train}-1}\sum |y_t-y_{t-1}|},
$$

where the denominator is computed from **Train₁**. Values below 1 indicate an improvement over the Naive Persistence baseline.

**Diebold–Mariano (DM) statistic** tests whether two forecasting models have significantly different prediction accuracy:

$$
DM=
\frac{\bar d}
{\sqrt{\widehat{\mathrm{Var}}(\bar d)}},
$$

where $\bar d = \overline{L(e_{1}) - L(e_{2})}$ is the mean loss differential between the reference model and the evaluated model. A statistically significant positive value indicates that the evaluated model outperforms the reference model.

### Establishing Predictability

The first scientific question is whether future geomagnetic activity is predictable at all, and if so, what the primary source of predictability is. Three increasingly informative baseline models are therefore evaluated:

1. **Naive Persistence**, which assumes that future geomagnetic conditions remain unchanged.
2. **Autoregressive Persistence**, which predicts future $D_{st}$ from its current value alone.
3. **Solar Wind Forcing**, which predicts future $D_{st}$ from the selected OMNI solar wind parameters.

Comparing these baselines establishes the incremental predictive value contributed by progressively richer sources of information. Only after this baseline is established is the contribution of neutron monitor observations evaluated in the following section.

| Variable                 | Role in the scientific validation                                                                                                    | First used by                |
| ------------------------ | ------------------------------------------------------------------------------------------------------------------------------------ | ---------------------------- |
| `feat`                   | Complete feature matrix containing predictors, targets, and timestamps from which all training and validation subsets are extracted. | Naive Persistence            |
| `masks`                  | Defines the chronological validation segments (`Val_Main`, `Val_Storm`) used consistently by every experiment.                       | Naive Persistence            |
| `BOUNDARIES`, `PURGE_H`  | Reconstruct the `Train_1` reference segment and enforce purge-aware temporal boundaries for metric computation.                      | Naive Persistence            |
| `train1_mask`, `y_train` | Provides the reference training series required to compute the MASE denominator for every model comparison.                          | Naive Persistence            |
| `K_HORIZONS`             | Defines the five forecast horizons (1, 3, 7, 12, 21 h) evaluated throughout all experiments.                                         | All models                   |
| `STORM_THR`              | Common storm threshold ($D_{st}<-50$ nT) used for Storm RMSE and weighted evaluation.                                                | All models                   |
| `SELECTED_FEATURES`      | Final predictor subset obtained from Feature Screening and used to construct the machine-learning feature sets.                      | Solar Wind Forcing (XGBoost) |
| `scaler`                 | Standardises the selected predictors before fitting machine-learning models. Not required by the persistence baselines.              | Solar Wind Forcing (XGBoost) |


In [5]:
# ── Scientific Validation Context ────────────────────────────────────────
feat_data         = pd.read_parquet('data/processed/feat_split.parquet')
fs                = joblib.load('models/feature_selection_results.pkl')
SELECTED_FEATURES = fs['selected_final']
scaler            = fs['scaler']
feature_sets      = build_feature_sets(SELECTED_FEATURES)

masks          = build_masks(feat_data['datetime'])
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']
train1_mask    = masks['train1']

y_train = feat_data.loc[train1_mask, 'dst'].copy()

EVAL_SEGMENTS = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

print(f"SELECTED_FEATURES : {len(SELECTED_FEATURES)}")
print(f"K_HORIZONS        : {K_HORIZONS}")
print(f"STORM_THR         : {STORM_THR} nT")
print(f"Train reference   : {train1_mask.sum():,} rows")
print(f"Val_Main          : {val_main_mask.sum():,} rows")
print(f"Val_Storm         : {val_storm_mask.sum():,} rows")

SELECTED_FEATURES : 24
K_HORIZONS        : [1, 3, 7, 12, 21]
STORM_THR         : -50.0 nT
Train reference   : 76,995 rows
Val_Main          : 52,542 rows
Val_Storm         : 1,446 rows


The four predictor configurations used in the ablation study are constructed here. `OMNI_FEATURES` is derived from `SELECTED_FEATURES` by excluding all neutron-derived variables, providing the solar-wind-only baseline. The remaining configurations extend this baseline by adding neutron representations one at a time, following the A→B→C→D experimental sequence.

In [6]:
# ── Feature set definitions ───────────────────────────────────────────────
from src.features import build_feature_sets, NEUTRON_ALL, NEUTRON_RAW, NEUTRON_DERIVED, NEUTRON_HISTORY

feature_sets      = build_feature_sets(SELECTED_FEATURES)
OMNI_FEATURES         = feature_sets['OMNI_FEATURES']
MODEL_A_OMNI          = feature_sets['MODEL_A_OMNI']
MODEL_B_OMNI_RAW      = feature_sets['MODEL_B_OMNI_RAW']
MODEL_C_OMNI_DNEUTRON = feature_sets['MODEL_C_OMNI_DNEUTRON']
MODEL_D_OMNI_HISTORY  = feature_sets['MODEL_D_OMNI_HISTORY']

print(f"OMNI features      : {len(MODEL_A_OMNI)}")
print(f"MODEL_B features   : {len(MODEL_B_OMNI_RAW)}")
print(f"MODEL_C features   : {len(MODEL_C_OMNI_DNEUTRON)}")
print(f"MODEL_D features   : {len(MODEL_D_OMNI_HISTORY)}")

OMNI features      : 20
MODEL_B features   : 21
MODEL_C features   : 21
MODEL_D features   : 23


#### Naive Persistence

The naïve persistence model assumes that the future geomagnetic state is identical to the current one:

$$
\hat{D}_{st}(t+h)=D_{st}(t).
$$

No learning is performed and no model parameters are estimated. The current observation is used directly as the prediction for every forecast horizon. Persistence therefore represents the minimum predictive baseline that any forecasting model must outperform.

For each forecast horizon $h \in \{1,3,7,12,21\}$, the forecast error is simply

$$
e_t^{(h)} = D_{st}(t+h)-D_{st}(t),
$$

which measures the natural evolution of the geomagnetic field over the prediction interval.

Persistence is evaluated independently on `Val_Main` (2009–2014) and `Val_Storm` (Halloween–November 2003). These results establish the reference RMSE, Storm RMSE, and MASE values against which all subsequent forecasting models are compared. By definition, the persistence model has $\mathrm{MASE}=1$, providing the natural reference point for forecast skill. Likewise, the Diebold–Mariano statistic is zero (p = 1.0) when persistence is compared with itself, serving as a sanity check for the evaluation pipeline.

The code below evaluates this baseline using the common validation setup defined above. `feat` provides both the current $D_{st}(t)$ values and the target columns `dst_target_{h}h`. `K_HORIZONS` ensures that the same five horizons are evaluated for every model. `EVAL_SEGMENTS` runs the test separately on `Val_Main` and `Val_Storm`, while `y_train` is passed only to compute the MASE reference scale. Since persistence is compared against itself, `y_persist_fn=None` and no additional persistence reference is required.

The persistence baseline requires no training, but it follows the same experimental protocol as all subsequent models. The reference series `y_train` is extracted from **Train1** only and is used to compute the MASE denominator, ensuring that forecast skill is measured relative to the same training period for every model. The baseline is then evaluated independently on `Val_Main` and `Val_Storm` across the five forecast horizons defined by `K_HORIZONS`.

Unlike persistence, the direct autoregressive baseline must be trained. Because the model predicts each forecast horizon directly rather than recursively, a separate linear model is fitted for every horizon in `K_HORIZONS`.

For horizon $h$, the model receives the current geomagnetic state `dst` as input and learns the mapping to the corresponding target `dst_target_{h}h` using **Train₁**. The fitted coefficients $(\alpha_h,\beta_h)$ therefore describe how the linear memory of the geomagnetic field changes with forecast horizon. The fitted coefficients are extracted for each horizon and stored separately. Their evolution with forecast horizon provides a direct interpretation of how rapidly the predictive influence of the current $D_{st}$ state decays.

In [7]:
# ── Naive Persistence Evaluation ──────────────────────────────────────────
# Evaluates the persistence baseline on all forecast horizons and both
# validation segments. Results are saved for later comparison with the
# autoregressive and machine-learning models.

persistence_model = NaivePersistence(dst_col="dst")

persistence_metrics = {
    seg_name: run_segment(
        model_name="naive_persistence",
        seg_name=seg_name,
        models={h: persistence_model for h in K_HORIZONS},
        X_seg_fn=lambda h: feat_data.loc[seg_mask],
        y_true_fn=lambda h: feat_data.loc[seg_mask, f"dst_target_{h}h"],
        y_train=y_train,
        y_persist_fn=None,
        storm_thr=STORM_THR,
        k_horizons=K_HORIZONS,
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(
    persistence_metrics,
    "models/metrics_naive_persistence.pkl"
)

print("Saved: models/metrics_naive_persistence.pkl")


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE=  3.55 | StormRMSE=  9.19 | MASE=0.754
  h= 3h | RMSE=  7.37 | StormRMSE= 22.34 | MASE=1.587
  h= 7h | RMSE= 10.90 | StormRMSE= 38.85 | MASE=2.300
  h=12h | RMSE= 13.27 | StormRMSE= 50.46 | MASE=2.774
  h=21h | RMSE= 15.44 | StormRMSE= 60.25 | MASE=3.240

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 10.04 | StormRMSE= 24.41 | MASE=1.626
  h= 3h | RMSE= 22.79 | StormRMSE= 58.54 | MASE=3.372
  h= 7h | RMSE= 38.76 | StormRMSE=102.29 | MASE=5.378
  h=12h | RMSE= 47.71 | StormRMSE=126.01 | MASE=6.792
  h=21h | RMSE= 54.42 | StormRMSE=142.70 | MASE=7.997
Saved: models/metrics_naive_persistence.pkl


> **Observations — Naive Persistence:**
>
> - Prediction accuracy decreases monotonically with forecast horizon on both validation segments, reflecting the gradual loss of autocorrelation in the $D_{st}$ series.
> - Performance is consistently poorer on `Val_Storm` than on `Val_Main`, indicating that persistence cannot capture the rapid evolution of extreme geomagnetic storms.
> - MASE is below 1 only at the 1-hour horizon on `Val_Main`. Beyond the shortest horizon, persistence no longer provides a competitive forecasting strategy.
> - These results establish the reference baseline against which the autoregressive and machine-learning models are compared.

#### Direct Autoregressive Baseline

The direct autoregressive baseline predicts future geomagnetic activity using only the current geomagnetic state:

$$
\hat{D}_{st}(t+h)=\alpha_h D_{st}(t)+\beta_h.
$$

A separate linear regression model is fitted for each forecast horizon using **Train₁** only. Unlike recursive autoregressive forecasting, the model predicts $D_{st}(t+h)$ directly in a single step, avoiding the accumulation of intermediate prediction errors.

This baseline quantifies the predictive information contained in the current $D_{st}$ value alone. Compared with persistence, the coefficients $\alpha_h$ and $\beta_h$ are estimated from the data rather than fixed to $(1,0)$, allowing the model to learn the average decay of geomagnetic disturbances.

Only the current $D_{st}(t)$ is used as a predictor. No additional lags or solar wind variables are included, ensuring that any subsequent improvement can be attributed to external solar wind forcing rather than a more complex autoregressive model.

The model is trained on **Train₁** and evaluated independently on `Val_Main` and `Val_Storm` using the same forecast horizons, metrics, and evaluation protocol as the persistence baseline. The fitted coefficients $\alpha_h$ and $\beta_h$ are retained to examine how the contribution of the current geomagnetic state changes with forecast horizon.

| Argument                                    | Meaning                     | Why this model needs it                                                      |
| ------------------------------------------- | --------------------------- | ---------------------------------------------------------------------------- |
| `DirectARBaseline(dst_col='dst')`           | Linear autoregressive model | Learns the relationship between the current and future (D_{st}).             |
| `feat_data.loc[train1_mask, ['dst']]`            | Predictor (D_{st}(t))       | The model uses only the current geomagnetic state.                           |
| `feat_data.loc[train1_mask, f'dst_target_{h}h']` | Target (D_{st}(t+h))        | A separate model is trained for each forecast horizon.                       |
| `train1_mask`                               | Training subset             | Ensures all baselines are fitted only on **Train₁**.                         |
| `K_HORIZONS`                                | Forecast horizons           | Creates five independent autoregressive models (1, 3, 7, 12, 21 h).          |
| `EVAL_SEGMENTS`                             | Validation subsets          | Evaluates every model on `Val_Main` and `Val_Storm` using the same protocol. |
| `y_train`                                   | Reference training series   | Required only for computing the MASE denominator.                            |
| `y_persist_fn`                              | Persistence forecast        | Provides the reference forecast for the Diebold–Mariano comparison.          |
| `extra_params_fn`                           | Returns `α` and `β`         | Saves the fitted autoregressive coefficients for later interpretation.       |


In [8]:
# ── Fit one model per horizon (Train_1 only) ─────────────────────────────
ar_models = {
    h: fit_model(
        DirectARBaseline(dst_col='dst'),
        feat_data.loc[train1_mask, ['dst']],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

ar_coefs = {h: {'alpha': ar_models[h].alpha_, 'beta': ar_models[h].beta_}
            for h in K_HORIZONS}

# ── Evaluate on both validation segments ─────────────────────────────────
ar_metrics = {
    seg_name: run_segment(
        model_name      = 'direct_ar',
        seg_name        = seg_name,
        models          = ar_models,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, ['dst']],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'alpha': round(m.alpha_, 4),
            'beta' : round(m.beta_,  4),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(ar_metrics, 'models/metrics_direct_ar.pkl')
print('\nSaved: models/metrics_direct_ar.pkl')


── Segment: val_main ──────────────────────────────────────
  alpha=0.977  beta=-0.380 | h= 1h | RMSE=  3.53 | StormRMSE=  9.26 | MASE=0.763
  alpha=0.902  beta=-1.612 | h= 3h | RMSE=  7.20 | StormRMSE= 22.49 | MASE=1.576
  alpha=0.774  beta=-3.731 | h= 7h | RMSE= 10.34 | StormRMSE= 38.08 | MASE=2.268
  alpha=0.660  beta=-5.622 | h=12h | RMSE= 12.25 | StormRMSE= 47.81 | MASE=2.727
  alpha=0.526  beta=-7.835 | h=21h | RMSE= 13.83 | StormRMSE= 54.98 | MASE=3.167

── Segment: val_storm ──────────────────────────────────────
  alpha=0.977  beta=-0.380 | h= 1h | RMSE= 10.00 | StormRMSE= 24.39 | MASE=1.605
  alpha=0.902  beta=-1.612 | h= 3h | RMSE= 22.26 | StormRMSE= 57.44 | MASE=3.157
  alpha=0.774  beta=-3.731 | h= 7h | RMSE= 36.15 | StormRMSE= 95.87 | MASE=4.747
  alpha=0.660  beta=-5.622 | h=12h | RMSE= 42.75 | StormRMSE=113.62 | MASE=5.761
  alpha=0.526  beta=-7.835 | h=21h | RMSE= 46.80 | StormRMSE=124.34 | MASE=6.555

Saved: models/metrics_direct_ar.pkl


> **Observations — Direct Autoregressive Baseline:**
>
> - The fitted autoregressive coefficient decreases monotonically with forecast horizon, from $\alpha=0.977$ at 1 h to $\alpha=0.526$ at 21 h. This indicates that the predictive influence of the current $D_{st}$ state gradually weakens as the forecast horizon increases, consistent with the decay of geomagnetic disturbances.
>
> - The Direct AR model consistently outperforms Naive Persistence on both validation segments. The improvement is modest on `Val_Main`, but becomes more pronounced on `Val_Storm`, particularly at longer forecast horizons.
>
> - The larger improvement during storm periods suggests that learning the average decay of the ring current is more informative than assuming a static geomagnetic state.
>
> - The Direct AR baseline provides the reference performance achievable using only the current geomagnetic state. Subsequent machine-learning models will be evaluated to determine whether solar wind parameters and neutron observations provide additional predictive information beyond this linear autoregressive memory.

#### Solar Wind Forcing (MODEL_A)

The previous experiment demonstrated the predictive information contained in the current geomagnetic state alone. The next question is whether the upstream solar wind provides additional predictive information beyond this linear autoregressive memory.

To answer this question, a nonlinear XGBoost model is trained using only the OMNI solar wind predictors:

$$
\hat{D}_{st}(t+h)=f_{\mathrm{XGB}}(\mathbf{x}_{\mathrm{OMNI}}(t)).
$$

Unlike the Direct AR baseline, which uses only the current $D_{st}$ value, XGBoost can model nonlinear relationships and interactions between multiple solar wind parameters. A separate model is trained for each forecast horizon using the 19 OMNI features retained during Feature Screening.

The machine-learning experiments are organised as a sequence of predefined feature sets. The same learning algorithm, training procedure, validation protocol, and evaluation metrics are used throughout. Only the predictor set changes between experiments, allowing the contribution of each additional source of information to be isolated.

- **MODEL_A** – OMNI solar wind features only.
- **MODEL_B** – OMNI features + raw neutron counts.
- **MODEL_C** – OMNI features + differential neutron flux ($\delta n$).
- **MODEL_D** – OMNI features + all selected neutron predictors.

In this experiment, **MODEL_A** establishes the nonlinear solar wind baseline. The remaining models will be compared against MODEL_A to determine whether neutron observations provide additional predictive information beyond the conventional OMNI measurements.

The models are trained on **Train₁** and evaluated on `Val_Main` and `Val_Storm` using the same protocol as the previous baselines. Storm sample weighting is applied during training to compensate for the severe imbalance between quiet and storm periods, ensuring that the model learns both regimes. Default XGBoost hyperparameters are used intentionally, as the objective is to evaluate the contribution of the predictor set rather than to maximise model performance. Hyperparameter optimisation is performed later, after the forecast horizon has been selected.

Before constructing the forecasting models, the selected predictors are grouped according to their physical origin. This allows the same modelling pipeline to evaluate progressively richer feature sets while changing only the neutron-related information.

- `NEUTRON_ALL` contains all neutron-monitor-derived predictors retained after Feature Screening.
- `NEUTRON_RAW` contains the instantaneous neutron count (`neutron_counts`).
- `NEUTRON_DERIVED` contains the differential neutron flux (`d_neutron`), the primary neutron representation investigated in this study.
- `NEUTRON_HISTORY` contains the lagged neutron observations representing the temporal evolution of the Forbush Decrease signal.

The remaining predictors form the conventional OMNI feature set:

- `OMNI_FEATURES` contains all selected predictors except the neutron-derived variables.

These groups are combined to construct the forecasting models used throughout the Scientific Validation stage. The first model,

- `MODEL_A_OMNI = OMNI_FEATURES`,

serves as the reference baseline containing only the conventional solar wind observations. Subsequent models progressively augment this baseline with neutron-derived predictors while leaving the learning algorithm, training procedure, and evaluation protocol unchanged.

A separate XGBoost model is fitted for each forecasting horizon using the `MODEL_A_OMNI` feature set defined in the Experimental Design. Each model is trained exclusively on **Train₁**, where the predictors are the selected OMNI features and the target is the corresponding future geomagnetic disturbance `dst_target_{h}h`.

The fitted models are subsequently evaluated on both validation segments (`Val_Main` and `Val_Storm`) using the common evaluation protocol established in the Experimental Design. The persistence forecast is supplied as the reference model for the Diebold–Mariano significance test, while the **Train₁** series is used to compute the MASE denominator.

For reproducibility, each fitted XGBoost pipeline is logged as an MLflow artifact and the evaluation metrics are stored in `metrics_xgb_model_a.pkl`, providing the reference nonlinear OMNI baseline for all subsequent neutron-feature experiments.

In [9]:
# ── Fit one pipeline per horizon (Train_1 only) ───────────────────────────
model_a_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst())]),
        feat_data.loc[train1_mask, MODEL_A_OMNI],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

# ── Evaluate on both validation segments ──────────────────────────────────
model_a_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_a_omni',
        seg_name        = seg_name,
        models          = model_a_pipes,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_A_OMNI],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_A_OMNI',
            'n_features' : len(MODEL_A_OMNI),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_a_metrics, 'models/metrics_xgb_model_a.pkl')
print('\nSaved: models/metrics_xgb_model_a.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 10.84 | StormRMSE= 18.49 | MASE=2.702
  h= 3h | RMSE= 11.54 | StormRMSE= 20.24 | MASE=2.818
  h= 7h | RMSE= 13.88 | StormRMSE= 29.32 | MASE=3.323
  h=12h | RMSE= 16.74 | StormRMSE= 38.60 | MASE=3.909
  h=21h | RMSE= 19.32 | StormRMSE= 49.38 | MASE=4.686

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 30.10 | StormRMSE= 78.69 | MASE=4.425
  h= 3h | RMSE= 33.51 | StormRMSE= 86.94 | MASE=4.883
  h= 7h | RMSE= 40.94 | StormRMSE=107.20 | MASE=5.620
  h=12h | RMSE= 45.09 | StormRMSE=118.19 | MASE=6.413
  h=21h | RMSE= 50.95 | StormRMSE=133.77 | MASE=7.321

Saved: models/metrics_xgb_model_a.pkl


> **Observations — XGBoost MODEL_A: OMNI only**
>
> - On `Val_Main`, MODEL_A improves storm-hour performance compared with the Direct AR baseline. At h=7h, Storm RMSE decreases from 38.08 nT to 29.32 nT, showing that nonlinear OMNI solar wind forcing adds predictive information beyond the current $D_{st}$ state alone.
>
> - Overall MASE remains above 1 at all horizons, meaning that the untuned OMNI XGBoost model does not yet outperform persistence in average absolute error. This is expected at this stage — MODEL_A serves as a controlled nonlinear OMNI baseline, not as the final tuned forecasting model.
>
> - On `Val_Storm`, MODEL_A underperforms both Persistence (Storm RMSE 102.29 nT) and the Direct AR baseline (95.87 nT) at h=7h, achieving Storm RMSE = 107.20 nT. This indicates that the Halloween 2003 superstorm ($D_{st}$ = −422 nT) lies outside the distribution of storms seen during training — an extrapolation regime where the untuned model degrades relative to simpler baselines.
>
> - These results establish the OMNI-only machine-learning reference against which neutron-augmented models are compared. The key question is whether adding neutron-monitor information improves this controlled baseline under the same training and evaluation protocol.

### Neutron Flux as a Predictive Signal

The primary scientific objective of this study is to determine whether neutron-monitor-derived features provide additional predictive information beyond the conventional OMNI solar wind parameters for multi-hour forecasting of the $D_{st}$ index. This section addresses that objective through a controlled comparison between equivalent models with and without neutron-derived predictors.

To ensure a fair comparison, all models are evaluated using the common experimental protocol established in the previous section, including identical chronological data splits, forecasting horizons, model architecture and evaluation metrics. Consequently, any observed differences in predictive performance can be attributed to the inclusion and representation of neutron-derived features rather than to differences in the modelling approach.

The section evaluates the proposed differential neutron flux model (MODEL_C), then compares alternative neutron feature representations through a controlled ablation study, and concludes with a feature importance analysis that quantifies the contribution of neutron-derived features across all forecasting horizons.

#### Experimental Design

The research objective is investigated through a controlled comparison of two equivalent supervised regression models. For each forecasting horizon $h$, both models estimate the future geomagnetic disturbance index

$$\hat{D}_{st}(t+h) = f(\mathbf{x}(t)),$$

where $\mathbf{x}(t)$ denotes the predictor vector available at time $t$. The learning algorithm, chronological training and validation splits, forecasting horizons and evaluation protocol remain identical. The only experimental variable is the composition of the predictor vector:

$$\begin{aligned}
\text{MODEL}_A &: \mathbf{x}(t) = \mathbf{x}_{\mathrm{OMNI}}(t), \\
\text{MODEL}_C &: \mathbf{x}(t) = \{\mathbf{x}_{\mathrm{OMNI}}(t),\, \delta n(t)\},
\end{aligned}$$

where $\delta n(t)$ is the differential neutron flux. By modifying only the predictor representation while keeping all other modelling components unchanged, the contribution of $\delta n(t)$ can be evaluated independently of the learning algorithm.

The implementation follows the common experimental protocol:

- **Training data:** Train₁ chronological segment.
- **Forecasting horizons:** $h \in \{1, 3, 7, 12, 21\}$ hours.
- **Learning algorithm:** XGBoost regressor with default hyperparameters.
- **Predictor set:** `MODEL_C_OMNI_DNEUTRON` — OMNI solar wind features augmented with $\delta n(t)$.
- **Target variable:** $D_{st}(t+h)$.
- **Evaluation segments:** `Val_Main` and `Val_Storm`.
- **Evaluation metrics:** $R^2$, RMSE, Storm RMSE and MASE.

In [10]:
MODEL_C_OMNI_DNEUTRON = OMNI_FEATURES + NEUTRON_DERIVED

# ── Fit one pipeline per forecasting horizon using Train_1 ────────────────
model_c_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst())]),
        feat_data.loc[train1_mask, MODEL_C_OMNI_DNEUTRON],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

# ── Evaluate on the validation segments ───────────────────────────────────
model_c_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_c_omni_dneutron',
        seg_name        = seg_name,
        models          = model_c_pipes,
        X_seg_fn        = lambda h, seg_mask=seg_mask: feat_data.loc[seg_mask, MODEL_C_OMNI_DNEUTRON],
        y_true_fn       = lambda h, seg_mask=seg_mask: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h, seg_mask=seg_mask: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_C_OMNI_DNEUTRON',
            'n_features' : len(MODEL_C_OMNI_DNEUTRON),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_c_metrics, 'models/metrics_xgb_model_c.pkl')
print('\nSaved: models/metrics_xgb_model_c.pkl')


── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 10.83 | StormRMSE= 18.87 | MASE=2.692
  h= 3h | RMSE= 11.49 | StormRMSE= 19.59 | MASE=2.818
  h= 7h | RMSE= 13.92 | StormRMSE= 28.86 | MASE=3.344
  h=12h | RMSE= 16.27 | StormRMSE= 38.90 | MASE=3.828
  h=21h | RMSE= 19.13 | StormRMSE= 49.68 | MASE=4.653

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 29.97 | StormRMSE= 78.17 | MASE=4.505
  h= 3h | RMSE= 32.51 | StormRMSE= 84.08 | MASE=4.788
  h= 7h | RMSE= 39.99 | StormRMSE=104.60 | MASE=5.522
  h=12h | RMSE= 45.32 | StormRMSE=118.60 | MASE=6.464
  h=21h | RMSE= 49.85 | StormRMSE=130.85 | MASE=7.069

Saved: models/metrics_xgb_model_c.pkl


> **Observations — XGBoost MODEL_C (OMNI + $\delta n$):**
>
> - MODEL_C exhibits the expected increase in forecasting error with increasing prediction horizon across both validation segments.
>
> - On `Val_Main`, the lowest Storm RMSE is obtained at h=1h (18.87 nT), with h=7h (28.86 nT) maintaining competitive storm forecasting performance at a substantially longer lead time. Overall RMSE increases monotonically with horizon, reflecting the progressive decay of the predictable signal.
>
> - On `Val_Storm`, Storm RMSE increases substantially with horizon — from 78.17 nT at h=1h to 104.60 nT at h=7h — reflecting the greater difficulty of predicting the extreme Halloween 2003 storm interval. At h=7h, MODEL_C achieves lower Storm RMSE than MODEL_A (104.60 vs 107.20 nT), with a positive $\Delta R^2$ = +0.0313, indicating that δn(t) provides measurable predictive information specifically during intense geomagnetic storm conditions.
>
> - The contrast between `Val_Main` (ΔR² = −0.0047 at h=7h) and `Val_Storm` (ΔR² = +0.0313 at h=7h) indicates that the contribution of δn(t) is storm-specific rather than general-purpose — consistent with the physical motivation that the Forbush Decrease precursor signal is most informative during intense geomagnetic activity.
>
> - A direct comparison with the remaining predictor configurations is presented in the following section.

In [11]:
# ── d_neutron feature importance по всички хоризонти ─────────────────────
print(f"{'h':>4} {'d_neutron rank':>16} {'gain %':>8} {'total features':>16}")
print("─" * 48)
for h in K_HORIZONS:
    booster = model_c_pipes[h].named_steps['model'].model_.get_booster()
    importance = booster.get_score(importance_type='gain')
    imp_df = pd.DataFrame.from_dict(importance, orient='index', columns=['gain'])
    imp_df = imp_df.sort_values('gain', ascending=False).reset_index()
    imp_df.columns = ['feature', 'gain']
    imp_df['rank'] = imp_df.index + 1
    imp_df['gain_pct'] = 100 * imp_df['gain'] / imp_df['gain'].sum()
    d = imp_df[imp_df['feature'] == 'd_neutron']
    if len(d) > 0:
        print(f"{h:>4} {d['rank'].values[0]:>16} {d['gain_pct'].values[0]:>8.2f}% {len(imp_df):>16}")
    else:
        print(f"{h:>4} {'NOT PRESENT':>16} {'0.00':>8}% {len(imp_df):>16}")

   h   d_neutron rank   gain %   total features
────────────────────────────────────────────────
   1               21     0.33%               21
   3               19     1.07%               21
   7               20     1.40%               21
  12               21     1.50%               21
  21               19     2.07%               21


#### Differential Neutron Flux Model

The predictive contribution of the differential neutron flux is evaluated by comparing MODEL_C with the OMNI-only reference model (MODEL_A). Since both models use the same learning algorithm, training procedure, forecasting horizons and evaluation protocol, the comparison isolates the effect of including the differential neutron flux in the predictor set.

The relative change in predictive performance is quantified as

$$\Delta R^2(h) = R^2(\text{MODEL\_C}) - R^2(\text{MODEL\_A})$$

where positive values indicate improved predictive performance after augmenting the OMNI predictor set with the differential neutron flux.

The comparison is performed for all forecasting horizons and for both validation segments (`Val_Main` and `Val_Storm`).

In [12]:
def get_model_metrics(metrics_dict, seg_name, h):
    """Extract RMSE and Storm RMSE for one model/segment/horizon."""
    m = metrics_dict[seg_name][h]
    return round(m['rmse'], 2), round(m['storm_rmse'], 2)

def build_comparison_row(seg_name, h):
    """
    Build one comparison row for horizon h across all four models.
    Returns dict ready for DataFrame construction.
    """
    p_rmse,  p_srmse  = get_model_metrics(persistence_metrics, seg_name, h)
    ar_rmse, ar_srmse = get_model_metrics(ar_metrics,          seg_name, h)
    a_rmse,  a_srmse  = get_model_metrics(model_a_metrics,     seg_name, h)
    c_rmse,  c_srmse  = get_model_metrics(model_c_metrics,     seg_name, h)

    return {
        'h'                    : h,
        'Persistence RMSE'     : p_rmse,
        'Persistence StormRMSE': p_srmse,
        'AR RMSE'              : ar_rmse,
        'AR StormRMSE'         : ar_srmse,
        'XGB-A RMSE'           : a_rmse,
        'XGB-A StormRMSE'      : a_srmse,
        'XGB-C RMSE'           : c_rmse,
        'XGB-C StormRMSE'      : c_srmse,
    }

def print_comparison_table(seg_name):
    """Build and print comparison table for one segment."""
    rows = [build_comparison_row(seg_name, h) for h in K_HORIZONS]
    df   = pd.DataFrame(rows).set_index('h')
    print(f'\n── {seg_name} ──────────────────────────────────────')
    print(df.to_string())

for seg_name in EVAL_SEGMENTS:
    print_comparison_table(seg_name)


── val_main ──────────────────────────────────────
    Persistence RMSE  Persistence StormRMSE  AR RMSE  AR StormRMSE  XGB-A RMSE  XGB-A StormRMSE  XGB-C RMSE  XGB-C StormRMSE
h                                                                                                                           
1               3.55                   9.19     3.53          9.26       10.84            18.49       10.83            18.87
3               7.37                  22.34     7.20         22.49       11.54            20.24       11.49            19.59
7              10.90                  38.85    10.34         38.08       13.88            29.32       13.92            28.86
12             13.27                  50.46    12.25         47.81       16.74            38.60       16.27            38.90
21             15.44                  60.25    13.83         54.98       19.32            49.38       19.13            49.68

── val_storm ──────────────────────────────────────
    Persistence RMSE

> **Observations — Comparison of Forecasting Models:**
>
> - Persistence and AR baselines achieve the lowest overall RMSE at short horizons on `Val_Main`, reflecting the strong temporal autocorrelation of $D_{st}$. At h=7h, Persistence RMSE = 10.90 nT and AR RMSE = 10.34 nT — both lower than XGBoost overall RMSE (13.88–13.92 nT). However, XGBoost substantially outperforms both baselines on Storm RMSE at h=7h (28.86–29.32 nT vs 38.08–38.85 nT), confirming that the nonlinear model captures storm-period dynamics that the linear baselines cannot.
>
> - On `Val_Main`, MODEL_C achieves lower Storm RMSE than MODEL_A at h=3h (19.59 vs 20.24 nT) and h=7h (28.86 vs 29.32 nT). At h=12h and h=21h the differences are negligible or reversed.
>
> - On `Val_Storm`, MODEL_C outperforms MODEL_A at h=3h (84.08 vs 86.94 nT), h=7h (104.60 vs 107.20 nT) and h=21h (130.85 vs 133.77 nT). However, both XGBoost models underperform the AR baseline on Storm RMSE at h=7h (104.60–107.20 nT vs 95.87 nT), and even underperform Persistence (102.29 nT). This indicates that the Halloween 2003 superstorm ($D_{st}$ = −422 nT) represents an extreme extrapolation regime that neither OMNI-only nor neutron-augmented XGBoost models can capture reliably at this stage — a finding that motivates the storm-specific weighting strategy applied in the Operational Forecasting Model.
>
> - The overall pattern indicates that δn(t) provides storm-specific rather than general-purpose predictive improvement, with the most consistent gains at h=3h and h=7h across both validation segments.

The coefficient of determination ($R^2$) is used to compare the explanatory performance of MODEL_A and MODEL_C across all forecasting horizons. The corresponding $\Delta R^2$ values are computed directly from the stored evaluation results for each validation segment, providing a compact summary of the relative predictive contribution of the differential neutron flux.

In [13]:
def compute_delta_r2(seg_name, h):
    """
    Extract the coefficient of determination (R²) for MODEL_A and MODEL_C
    together with their difference (ΔR²) for a given validation segment
    and forecasting horizon.
    """
    r2_a = model_a_metrics[seg_name][h]['r2']
    r2_c = model_c_metrics[seg_name][h]['r2']
    return r2_a, r2_c, r2_c - r2_a


def print_delta_r2_table(seg_name):
    """
    Print the R² comparison table for all forecasting horizons within
    the selected validation segment.
    """
    print(f'\nΔR² = R²(OMNI+δn) - R²(OMNI)  [{seg_name}]')
    print('=' * 45)
    print(f'{"h":>4} | {"R²(A)":>8} | {"R²(C)":>8} | {"ΔR²":>8}')
    print('-' * 45)
    for h in K_HORIZONS:
        r2_a, r2_c, delta = compute_delta_r2(seg_name, h)
        print(f'{h:>4}h | {r2_a:>8.4f} | {r2_c:>8.4f} | {delta:>+8.4f}')

for seg_name in EVAL_SEGMENTS:
    print_delta_r2_table(seg_name)



ΔR² = R²(OMNI+δn) - R²(OMNI)  [val_main]
   h |    R²(A) |    R²(C) |      ΔR²
---------------------------------------------
   1h |   0.5160 |   0.5166 |  +0.0006
   3h |   0.4518 |   0.4562 |  +0.0045
   7h |   0.2061 |   0.2014 |  -0.0047
  12h |  -0.1546 |  -0.0906 |  +0.0641
  21h |  -0.5380 |  -0.5081 |  +0.0299

ΔR² = R²(OMNI+δn) - R²(OMNI)  [val_storm]
   h |    R²(A) |    R²(C) |      ΔR²
---------------------------------------------
   1h |   0.6311 |   0.6344 |  +0.0034
   3h |   0.5428 |   0.5697 |  +0.0268
   7h |   0.3178 |   0.3491 |  +0.0313
  12h |   0.1725 |   0.1639 |  -0.0085
  21h |  -0.0560 |  -0.0110 |  +0.0449


> **Observations — Relative Performance ($\Delta R^2$):**
> - On `Val_Main`, positive $\Delta R^2$ values are obtained at h=1, 3, 12 and 21 hours. At h=7h a small negative value (−0.0047) indicates that δn(t) does not improve overall explained variance on the standard validation segment at the primary horizon.
> - On `Val_Storm`, positive $\Delta R^2$ values are obtained at h=1, 3, 7 and 21 hours. Notably, h=7h shows $\Delta R^2$ = +0.0313 — the neutron-derived feature improves explanatory performance specifically during the extreme Halloween 2003 storm interval, consistent with the physical expectation that the Forbush Decrease precursor signal is most informative during intense geomagnetic activity.
> - The contrast between val_main (ΔR²=−0.0047 at h=7h) and val_storm (ΔR²=+0.0313 at h=7h) indicates that δn(t) provides storm-specific rather than general-purpose predictive information at the primary forecasting horizon.
> - The magnitude of the observed changes remains modest, suggesting that the differential neutron flux provides incremental rather than dominant predictive information — a finding consistently supported by the ablation study and feature importance analysis.

#### Ablation Study

The ablation study investigates the predictive contribution of neutron-monitor information by separating three independent questions:

1. **Does neutron information provide additional predictive value beyond OMNI?** MODEL_A → MODEL_B introduces the raw neutron count rate $n(t)$ while keeping all other features unchanged.
2. **Which neutron representation captures the signal more effectively?** MODEL_B → MODEL_C compares the raw count rate with the differential flux $\delta n(t)$, isolating the effect of the feature transformation.
3. **Does neutron history provide additional predictive value beyond the current signal?** MODEL_C → MODEL_D evaluates whether lagged neutron counts ($n_{t-3h}$, $n_{t-7h}$) add information beyond the current differential flux.

This sequential design isolates one experimental factor at a time. MODEL_A and MODEL_C results are carried forward from the preceding experiments. MODEL_B and MODEL_D are trained in this section under the same XGBoost configuration, training split and evaluation protocol — the neutron representation is the only difference between the four configurations.

In [14]:
# ── Cell: Ablation Study — MODEL_B and MODEL_D ───────────────────────────
# MODEL_A and MODEL_C metrics already computed in Stages 3-4.
# Only MODEL_B (OMNI + raw neutron) and MODEL_D (OMNI + δn + lags) are new.

# ── Fit MODEL_B (OMNI + raw neutron counts) ──────────────────────────────
print('\n── MODEL_B: OMNI + raw neutron counts ───────────────────────────────')
model_b_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst())]),
        feat_data.loc[train1_mask, MODEL_B_OMNI_RAW],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

model_b_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_b_omni_raw',
        seg_name        = seg_name,
        models          = model_b_pipes,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_B_OMNI_RAW],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_B_OMNI_RAW',
            'n_features' : len(MODEL_B_OMNI_RAW),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_b_metrics, 'models/metrics_xgb_model_b.pkl')
print('Saved: models/metrics_xgb_model_b.pkl')


── MODEL_B: OMNI + raw neutron counts ───────────────────────────────

── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 11.31 | StormRMSE= 18.93 | MASE=2.838
  h= 3h | RMSE= 12.49 | StormRMSE= 19.87 | MASE=3.057
  h= 7h | RMSE= 14.80 | StormRMSE= 28.77 | MASE=3.493
  h=12h | RMSE= 16.74 | StormRMSE= 40.06 | MASE=3.848
  h=21h | RMSE= 17.26 | StormRMSE= 52.08 | MASE=4.106

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 30.82 | StormRMSE= 80.88 | MASE=4.486
  h= 3h | RMSE= 32.69 | StormRMSE= 83.02 | MASE=5.194
  h= 7h | RMSE= 41.04 | StormRMSE=106.98 | MASE=5.894
  h=12h | RMSE= 44.97 | StormRMSE=115.56 | MASE=6.972
  h=21h | RMSE= 49.70 | StormRMSE=126.94 | MASE=8.016
Saved: models/metrics_xgb_model_b.pkl


In [15]:
# ── Fit MODEL_D (OMNI + δn + neutron lags) ───────────────────────────────
print('\n── MODEL_D: OMNI + δn + neutron lags ───────────────────────────────')
model_d_pipes = {
    h: fit_model(
        Pipeline([('model', XGBoostDst())]),
        feat_data.loc[train1_mask, MODEL_D_OMNI_HISTORY],
        feat_data.loc[train1_mask, f'dst_target_{h}h'],
    )
    for h in K_HORIZONS
}

model_d_metrics = {
    seg_name: run_segment(
        model_name      = 'xgb_model_d_full_neutron',
        seg_name        = seg_name,
        models          = model_d_pipes,
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_D_OMNI_HISTORY],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, f'dst_target_{h}h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = K_HORIZONS,
        extra_params_fn = lambda h, m: {
            'feature_set': 'MODEL_D_OMNI_HISTORY',
            'n_features' : len(MODEL_D_OMNI_HISTORY),
        },
    )
    for seg_name, seg_mask in EVAL_SEGMENTS.items()
}

joblib.dump(model_d_metrics, 'models/metrics_xgb_model_d.pkl')
print('Saved: models/metrics_xgb_model_d.pkl')


── MODEL_D: OMNI + δn + neutron lags ───────────────────────────────

── Segment: val_main ──────────────────────────────────────
  h= 1h | RMSE= 11.09 | StormRMSE= 18.64 | MASE=2.772
  h= 3h | RMSE= 11.46 | StormRMSE= 19.83 | MASE=2.798
  h= 7h | RMSE= 14.55 | StormRMSE= 29.91 | MASE=3.496
  h=12h | RMSE= 16.25 | StormRMSE= 42.19 | MASE=3.836
  h=21h | RMSE= 17.06 | StormRMSE= 53.29 | MASE=4.028

── Segment: val_storm ──────────────────────────────────────
  h= 1h | RMSE= 29.45 | StormRMSE= 76.14 | MASE=4.539
  h= 3h | RMSE= 33.06 | StormRMSE= 85.09 | MASE=4.995
  h= 7h | RMSE= 40.19 | StormRMSE=102.77 | MASE=6.133
  h=12h | RMSE= 43.89 | StormRMSE=112.25 | MASE=6.850
  h=21h | RMSE= 51.11 | StormRMSE=127.39 | MASE=8.467
Saved: models/metrics_xgb_model_d.pkl


In [16]:
# ── Ablation Comparison — Storm RMSE, all models and horizons ─────────────
MODELS = {
    'A': model_a_metrics,
    'B': model_b_metrics,
    'C': model_c_metrics,
    'D': model_d_metrics,
}

SEGMENTS = {'storm': 'val_storm', 'quiet': 'val_main'}

def get_segment_metrics(metrics, seg, metric_key):
    return [metrics[seg][h][metric_key] for h in K_HORIZONS]

def get_model_comparison(label, metrics, metric_key):
    return {
        f'{label}_{col}': get_segment_metrics(metrics, seg, metric_key)
        for col, seg in SEGMENTS.items()
    }

def build_comparison(metric_key='storm_rmse'):
    data = {}
    for label, metrics in MODELS.items():
        data.update(get_model_comparison(label, metrics, metric_key))
    return pd.DataFrame(data, index=K_HORIZONS)

def delta_vs_a(metrics, h, seg):
    """Storm RMSE improvement vs MODEL_A."""
    return model_a_metrics[seg][h]['storm_rmse'] - metrics[seg][h]['storm_rmse']

storm_rmse_table = build_comparison('storm_rmse')
print("── Storm RMSE — all models, both segments ───────────────────────────")
print(storm_rmse_table.round(2))
print()

print(f"{'h':>4} {'B storm Δ':>10} {'B quiet Δ':>10} {'C storm Δ':>10} {'C quiet Δ':>10} {'D storm Δ':>10} {'D quiet Δ':>10}")
print("─" * 68)
for h in K_HORIZONS:
    vals = [(delta_vs_a(MODELS[m], h, 'val_storm'), 
             delta_vs_a(MODELS[m], h, 'val_main')) for m in ['B', 'C', 'D']]
    flat = [v for pair in vals for v in pair]
    print(f"{h:>4} " + " ".join(f"{v:>+10.2f}" for v in flat))

joblib.dump({'storm_rmse_table': storm_rmse_table}, 'models/ablation_comparison.pkl')
print('\nSaved: models/ablation_comparison.pkl')

── Storm RMSE — all models, both segments ───────────────────────────
    A_storm  A_quiet  B_storm  B_quiet  C_storm  C_quiet  D_storm  D_quiet
1     78.69    18.49    80.88    18.93    78.17    18.87    76.14    18.64
3     86.94    20.24    83.02    19.87    84.08    19.59    85.09    19.83
7    107.20    29.32   106.98    28.77   104.60    28.86   102.77    29.91
12   118.19    38.60   115.56    40.06   118.60    38.90   112.25    42.19
21   133.77    49.38   126.94    52.08   130.85    49.68   127.39    53.29

   h  B storm Δ  B quiet Δ  C storm Δ  C quiet Δ  D storm Δ  D quiet Δ
────────────────────────────────────────────────────────────────────
   1      -2.19      -0.44      +0.52      -0.38      +2.55      -0.15
   3      +3.93      +0.37      +2.86      +0.65      +1.85      +0.41
   7      +0.22      +0.55      +2.59      +0.46      +4.43      -0.59
  12      +2.63      -1.46      -0.42      -0.31      +5.93      -3.59
  21      +6.82      -2.70      +2.92      -0.30      +

> **Observations — Ablation Study:**
>
> The storm-period improvements increase with forecasting horizon across all neutron representations — consistent with the physical expectation that the Forbush Decrease precursor signal becomes relatively more important as OMNI solar wind predictability decreases at longer lead times.
>
> On `Val_Storm`, MODEL_D achieves the largest storm-period improvements (+4.43 nT at h=7h, +5.93 nT at h=12h, +6.38 nT at h=21h). MODEL_C provides consistent gains at h=3h (+2.86 nT) and h=7h (+2.59 nT). MODEL_B underperforms MODEL_A at h=1h (−2.19 nT) but recovers at longer horizons.
>
> On `Val_Main`, differences between models are small (within ±1.5 nT at h≤7h). At longer horizons, MODEL_B and MODEL_D show notable quiet-period degradation, while MODEL_C remains close to MODEL_A across all horizons.
>
> **Model selection:** MODEL_C offers the most balanced trade-off — consistent storm-period improvement at h=3h and h=7h with a near-negligible quiet-period cost (−0.99 nT cumulative). MODEL_D achieves stronger storm performance but at the cost of quiet-period reliability. MODEL_C is therefore carried forward as the neutron representation. The forecasting horizon is determined in the Feature Importance Summary below.

### Feature Importance Summary

Gain-based feature importance is extracted from the fitted XGBoost models for all four predictor configurations across all five forecasting horizons. Gain measures the average improvement in the loss function when a feature is used in a tree split — a feature with near-zero gain is rarely selected and contributes minimally to the model's predictions regardless of its presence in the feature set.

This analysis serves two purposes: it quantifies whether neutron-derived features are actively used by the model, and it identifies the forecasting horizon at which the neutron signal provides the greatest relative contribution compared with the dominant OMNI features.

The table below shows gain-based feature importance for the neutron-derived features across all four models and five forecasting horizons. Gain measures the average improvement in the loss function when a feature is used in a tree split. The equal-share baseline (~4.76% for MODEL_B/C, ~4.35% for MODEL_D) represents the expected gain if all features contributed equally — values below this indicate the feature is used less than average.

In [17]:
# ── Cross table: neutron gain % по модел и хоризонт ──────────────────────
def get_gain_pct(pipes, h, feature):
    """Return gain % for a single feature in a model at horizon h."""
    booster = pipes[h].named_steps['model'].model_.get_booster()
    imp = pd.DataFrame.from_dict(
        booster.get_score(importance_type='gain'),
        orient='index', columns=['gain']
    )
    imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
    d = imp[imp.index == feature]
    return round(d['gain_pct'].values[0], 2) if len(d) > 0 else 0.0


cross = pd.DataFrame({
    'MODEL_B (neutron_counts)' : [get_gain_pct(model_b_pipes, h, 'neutron_counts') for h in K_HORIZONS],
    'MODEL_C (d_neutron)'      : [get_gain_pct(model_c_pipes, h, 'd_neutron') for h in K_HORIZONS],
    'MODEL_D (lag3)'           : [get_gain_pct(model_d_pipes, h, 'neutron_counts_lag3') for h in K_HORIZONS],
    'MODEL_D (lag7)'           : [get_gain_pct(model_d_pipes, h, 'neutron_counts_lag7') for h in K_HORIZONS],
}, index=K_HORIZONS)
cross.index.name = 'h'


print("── Neutron feature gain % by model and horizon ──────────────────────")
print(f"{'':>4} {'MODEL_B':>20} {'MODEL_C':>15} {'MODEL_D lag3':>14} {'MODEL_D lag7':>14}")
print(f"{'h':>4} {'neutron_counts':>20} {'d_neutron':>15} {'lag3':>14} {'lag7':>14}")
print("─" * 67)
for h in [1, 3, 7, 12, 21]:
    print(f"{h:>4} {cross.loc[h,'MODEL_B (neutron_counts)']:>20.2f}% "
          f"{cross.loc[h,'MODEL_C (d_neutron)']:>14.2f}% "
          f"{cross.loc[h,'MODEL_D (lag3)']:>13.2f}% "
          f"{cross.loc[h,'MODEL_D (lag7)']:>13.2f}%")


── Neutron feature gain % by model and horizon ──────────────────────
                  MODEL_B         MODEL_C   MODEL_D lag3   MODEL_D lag7
   h       neutron_counts       d_neutron           lag3           lag7
───────────────────────────────────────────────────────────────────
   1                 1.17%           0.33%          1.06%          0.86%
   3                 1.28%           1.07%          0.89%          1.06%
   7                 2.04%           1.40%          1.45%          1.73%
  12                 3.44%           1.50%          3.61%          3.41%
  21                 6.58%           2.07%          5.41%          5.86%


The second table contextualises the neutron feature gain relative to the dominant OMNI predictor at each horizon. This shows how much of the total gain is captured by the top OMNI feature compared with the best neutron representation in each model.

In [18]:
def get_gain_pct(pipes, h, feature):
    booster = pipes[h].named_steps['model'].model_.get_booster()
    imp = pd.DataFrame.from_dict(
        booster.get_score(importance_type='gain'),
        orient='index', columns=['gain']
    )
    imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
    d = imp[imp.index == feature]
    return round(d['gain_pct'].values[0], 2) if len(d) > 0 else 0.0

def get_top_omni(pipes, h):
    booster = pipes[h].named_steps['model'].model_.get_booster()
    imp = pd.DataFrame.from_dict(
        booster.get_score(importance_type='gain'),
        orient='index', columns=['gain']
    )
    imp['gain_pct'] = 100 * imp['gain'] / imp['gain'].sum()
    imp = imp.sort_values('gain_pct', ascending=False)
    top = imp.iloc[0]
    return top.name, top['gain_pct']

def get_equal_share(pipes, h):
    booster = pipes[h].named_steps['model'].model_.get_booster()
    return 100 / len(booster.get_score(importance_type='gain'))

def get_equal_share(pipes, h):
    """Return equal-share baseline % for this model at horizon h."""
    booster = pipes[h].named_steps['model'].model_.get_booster()
    n_features = len(booster.get_score(importance_type='gain'))
    return 100 / n_features

# ── Cross table ───────────────────────────────────────────────────────────
print(f"{'h':>4} {'Top OMNI feature':>20} {'OMNI gain%':>11} "
      f"{'neutron_counts (B)':>20} {'d_neutron (C)':>15} {'lag7 (D)':>10}")
print("─" * 85)

for h in K_HORIZONS:
    feat_name, omni_pct = get_top_omni(model_a_pipes, h)
    nc   = get_gain_pct(model_b_pipes, h, 'neutron_counts')
    dn   = get_gain_pct(model_c_pipes, h, 'd_neutron')
    lag7 = get_gain_pct(model_d_pipes, h, 'neutron_counts_lag7')
    eq_b = get_equal_share(model_b_pipes, h)
    eq_c = get_equal_share(model_c_pipes, h)
    eq_d = get_equal_share(model_d_pipes, h)
    print(f"{h:>4} {feat_name:>20} {omni_pct:>10.2f}% {nc:>20} {dn:>15} {lag7:>10}")

print(f"\n  Equal-share baseline: MODEL_B/C ~{eq_b:.2f}%, MODEL_D ~{eq_d:.2f}%")

   h     Top OMNI feature  OMNI gain%   neutron_counts (B)   d_neutron (C)   lag7 (D)
─────────────────────────────────────────────────────────────────────────────────────
   1           bz_acc_12h      28.23%                 1.17            0.33       0.86
   3           bz_acc_12h      26.28%                 1.28            1.07       1.06
   7           bz_acc_12h      21.11%                 2.04             1.4       1.73
  12           bz_acc_12h      15.73%                 3.44             1.5       3.41
  21           bz_acc_12h      12.25%                 6.58            2.07       5.86

  Equal-share baseline: MODEL_B/C ~4.76%, MODEL_D ~4.35%


The table below shows the feature selection status for all neutron-derived features — whether each passed the strict voting criterion (`selected_strict`, both methods agree) or the relaxed threshold (`selected_final`, at least one method), and which predictor configuration it is used in.

In [19]:
# ── Feature Selection consistency check ───────────────────────────────────
print("Feature Selection status — neutron features:")
print(f"{'Feature':>25} {'strict':>8} {'final':>8} {'Used in':>15}")
print("─" * 60)

neutron_features = {
    'neutron_counts'       : 'MODEL_B',
    'd_neutron'            : 'MODEL_C, MODEL_D',
    'neutron_counts_lag1'  : '—',
    'neutron_counts_lag3'  : 'MODEL_D',
    'neutron_counts_lag7'  : 'MODEL_D',
    'neutron_counts_lag12' : '—',
    'neutron_counts_lag21' : '—',
}

for feature, used_in in neutron_features.items():
    strict = feature in fs['selected_strict']
    final  = feature in fs['selected_final']
    print(f"  {feature:>25}: {str(strict):>8} {str(final):>8} {used_in:>15}")

Feature Selection status — neutron features:
                  Feature   strict    final         Used in
────────────────────────────────────────────────────────────
             neutron_counts:     True     True         MODEL_B
                  d_neutron:    False     True MODEL_C, MODEL_D
        neutron_counts_lag1:    False    False               —
        neutron_counts_lag3:     True     True         MODEL_D
        neutron_counts_lag7:     True     True         MODEL_D
       neutron_counts_lag12:    False    False               —
       neutron_counts_lag21:    False    False               —


In [22]:
# ── Diebold-Mariano тест: MODEL_C vs MODEL_A — val_main и val_storm ───────
from src.evaluate import diebold_mariano

def dm_test(pipes_a, pipes_c, features_a, features_c, mask, h):
    """
    One-sided DM test: positive stat = MODEL_C better than MODEL_A.
    d = L(e_A) - L(e_C), positive d_bar → MODEL_C has lower loss.
    """
    y_true = feat_data.loc[mask, f'dst_target_{h}h'].dropna()
    idx    = y_true.index
    pred_a = pipes_a[h].predict(feat_data.loc[idx, features_a])
    pred_c = pipes_c[h].predict(feat_data.loc[idx, features_c])
    stat, _ = diebold_mariano(y_true.values, pred_a, pred_c, h=h)
    pval    = 1 - stats.norm.cdf(stat)  # one-sided: P(Z > stat)
    return stat, pval

print(f"{'h':>4} {'val_main DM':>12} {'val_main p':>12} {'sig':>5} "
      f"{'val_storm DM':>14} {'val_storm p':>12} {'sig':>5}")
print("─" * 70)

for h in K_HORIZONS:
    dm_m, p_m = dm_test(model_a_pipes, model_c_pipes,
                         OMNI_FEATURES, MODEL_C_OMNI_DNEUTRON,
                         val_main_mask, h)
    dm_s, p_s = dm_test(model_a_pipes, model_c_pipes,
                         OMNI_FEATURES, MODEL_C_OMNI_DNEUTRON,
                         val_storm_mask, h)
    sig_m = "+" if p_m < 0.05 else "—"
    sig_s = "+" if p_s < 0.05 else "—"
    print(f"{h:>4} {dm_m:>12.3f} {p_m:>12.4f} {sig_m:>5} "
          f"{dm_s:>14.3f} {p_s:>12.4f} {sig_s:>5}")

   h  val_main DM   val_main p   sig   val_storm DM  val_storm p   sig
──────────────────────────────────────────────────────────────────────
   1        0.614       0.2696     —          0.530       0.2981     —
   3        3.716       0.0001     +          4.202       0.0000     +
   7       -1.722       0.9575     —          3.653       0.0001     +
  12       17.388       0.0000     +         -1.635       0.9490     —
  21        7.193       0.0000     +          4.463       0.0000     +


> **Observations — Diebold-Mariano Test (MODEL_C vs MODEL_A):**
>
> The DM test evaluates whether the squared forecast error of MODEL_C is statistically significantly lower than that of MODEL_A (one-sided, $p < 0.05$).
>
> At h=3h, MODEL_C significantly outperforms MODEL_A on both validation segments (val_main: $p=0.0001$, val_storm: $p<0.0001$). The statistically significant improvement on val_main is consistent with the large sample size (52,542 rows) — a systematic Storm RMSE improvement of ~0.65 nT is sufficient to produce a significant test statistic at this scale. This is distinct from practical significance: feature importance analysis shows that $\delta n(t)$ receives only 1.07% of total gain at h=3h, well below the equal-share baseline of ~4.76%.
>
> At h=7h, the result is asymmetric: MODEL_C does not significantly outperform MODEL_A on val_main ($p=0.96$, DM=−1.72), but significantly outperforms it on val_storm ($p=0.0001$, DM=+3.65). This indicates that the predictive contribution of $\delta n(t)$ at the primary forecasting horizon is storm-specific rather than general-purpose — consistent with the feature importance result showing that neutron gain is highest at h=7h among operationally viable horizons, yet concentrated in the extreme storm regime rather than distributed across all conditions.
>
> At h=12h and h=21h, significance is observed but both models have negative $R^2$, limiting operational relevance.
>
> Taken together, the DM test and feature importance analysis tell a consistent story: $\delta n(t)$ provides a weak but genuine predictive signal that is statistically detectable at h=3h due to sample size, and physically meaningful at h=7h specifically during extreme geomagnetic storm conditions.

**Methodological note:** The neutron representation is selected independently of the operating horizon in order to identify the feature configuration that provides the most consistent performance across the full forecasting range. The operating horizon is then determined from the performance profile of the selected representation. This sequential design isolates the contribution of the neutron-derived features from the choice of forecasting horizon. A joint optimisation over the combined (representation, horizon) space could identify a single best-performing combination, but would not distinguish whether the observed improvement originates from the neutron representation itself or from the selected forecasting horizon, which is the primary objective of the ablation study.

> **Observations — Feature Importance Summary:**
>
> `bz_acc_12h` is the dominant predictor at every horizon, accounting for 28.23% of total gain at h=1h and declining to 12.25% at h=21h. Even at its lowest, it remains 6–10× more informative than any neutron-derived feature.
>
> All neutron features remain well below the equal-share baseline (~4.76% for MODEL_B/C, ~4.35% for MODEL_D) at every operationally viable horizon (h=1, 3, 7). At h=7h: `neutron_counts` (MODEL_B) 2.04%, `neutron_counts_lag7` (MODEL_D) 1.73%, `d_neutron` (MODEL_C) 1.40%. The signal is present but weak relative to the dominant OMNI predictors.
>
> A consistent horizon-dependent pattern emerges: neutron gain increases monotonically with forecasting horizon. At h=21h, `neutron_counts` reaches 6.58% and `neutron_counts_lag7` reaches 5.86% — both exceeding the equal-share baseline. However, h=21h corresponds to negative $R^2$ for all models, making it operationally unsuitable.
>
> **Note on feature selection:** `neutron_counts_lag3` and `neutron_counts_lag7` passed the strict voting criterion (`selected_strict`) and are used in MODEL_D. `d_neutron` passed only the relaxed threshold (`selected_final`) and is used in MODEL_C. `neutron_counts_lag1`, `neutron_counts_lag12` and `neutron_counts_lag21` did not pass either criterion and are absent from all models. The feature selection procedure therefore independently supports the finding that neutron-derived features carry a weak but genuine signal.
>
> Among the viable horizons (h=1, 3, 7), h=7h shows the highest neutron gain across all representations. This provides empirical support for h=7h as the primary forecasting horizon, complementing the physical motivation from the Burton ring-current decay timescale `[BUR75]` and the 7–21h precursor interval identified by Kisvárdai et al. `[KIS25]`.
>
> **Horizon selection:** $h^* = 7$h. While h=3h provides the strongest statistical performance (DM test significant on both segments), the 7-hour horizon offers greater operational lead time, shows the highest neutron signal gain among viable horizons, and aligns with the physical timescales of geomagnetic storm evolution. MODEL_C (OMNI + $\delta n$) at $h^* = 7$h is carried forward to the Operational Forecasting Model.

## Operational Forecasting Model

With $h^* = 7$h and MODEL_C (OMNI + $\delta n$) fixed from the scientific validation, the forecasting task is reduced to a single operational problem: optimise and compare model architectures for one horizon and one feature set.

**Candidate models: XGBoost and LightGBM**

Both candidate models are selected on three criteria: suitability for tabular time-series forecasting, native support for storm sample weighting via `sample_weight` in `fit()`, and comparable architecture that enables a fair performance comparison. XGBoost grows trees level-wise by default; LightGBM uses a leaf-wise strategy that often achieves lower loss with fewer iterations. This architectural difference means the two models may generalise differently on storm-dominated data — making the comparison informative rather than redundant.

**Pipeline:**

1. Both models are retrained on Train_1 + Train_2 with a reference hyperparameter configuration to establish a baseline.
2. Hyperparameters are optimised via RandomizedSearchCV with TimeSeriesSplit (5 folds) to minimise Storm RMSE.
3. Tuned models are evaluated on Val_Main and Val_Storm to assess generalisation under routine and extreme conditions.
4. Storm weighting is applied to address class imbalance between storm and quiet hours.
5. Residuals of the best model are inspected via ACF/PACF to assess whether systematic structure remains.
6. The final model is selected based on combined Val_Main and Val_Storm performance.

**Final evaluation**

The held-out test set (Test_Active: 2015, Test_Quiet: 2016–2023) serves as an independent estimate of operational performance on unseen data from Solar Cycles 24–25. Test_Active and Test_Quiet are evaluated separately to assess whether performance generalises across different solar activity levels. The test set is evaluated exactly once, after all modelling decisions are finalised.

### Candidate Model Training

Following the scientific validation, both the forecasting horizon ($h^* = 7$h) and the feature set (MODEL_C, OMNI + $\delta n$, 21 features) are fixed and are no longer modified. Consequently, this stage focuses exclusively on model optimisation rather than feature engineering or horizon selection.

Both models are trained on the full training set (Train_1 + Train_2, 1995–2008) with the reference hyperparameter configuration defined below. The initial models establish a reference performance prior to hyperparameter optimisation.

**Why Train_1 + Train_2:** during the scientific validation Train_2 was deliberately withheld to keep the horizon and feature set selection clean. Now that both decisions are fixed, all available training data can be used — Train_2 adds Solar Cycle 23 declining phase (2004–2008) which contains storm variety not present in Train_1 alone.

**The core idea — gradient boosting:**

Both models build the forecast step by step. At each step a new tree corrects the mistakes of all previous trees. The final forecast is the sum of all trees:

$$\hat{D}_{st}(t+7) = \sum_{k=1}^{K} f_k(\mathbf{x}_t)$$

where each $f_k$ fits the negative gradient of the loss from the previous step.

**Where XGBoost and LightGBM differ:**

XGBoost grows trees level-wise and applies explicit regularisation `[CHE16]`:

$$\mathcal{L}^{(k)} = \sum_t w_t l\!\left(y_t,\, \hat{y}_t^{(k-1)} + f_k(\mathbf{x}_t)\right) + \gamma T + \tfrac{1}{2}\lambda \|w\|^2$$

LightGBM uses a leaf-wise strategy — always splitting whichever leaf gives the biggest loss reduction — and applies GOSS and EFB to speed up training `[KE17]`.

**Reference hyperparameter configuration:**

| Parameter | XGBoost | LightGBM | Effect |
|---|---|---|---|
| `n_estimators` | 500 | 500 | Number of trees |
| `max_depth` | 5 | 5 | Maximum tree depth |
| `learning_rate` | 0.05 | 0.05 | Step size |
| `num_leaves` | — | 31 | LightGBM leaf-wise complexity |
| `subsample` | 0.8 | 0.8 | Row subsampling per tree |
| `colsample_bytree` | 0.8 | 0.8 | Feature subsampling per tree |
| `reg_alpha / lambda` | 0 / 1 | 0 / 1 | L1 / L2 regularisation |

Storm sample weighting is described in the Storm Weighting Strategy section below.

In [ ]:
train2_mask    = segment_mask('train2_start', 'train2_end',
                               purge_start=True,  purge_end=True)
train_mask     = masks['train']       # Train_1 | Train_2 — for Stage 6
val_main_mask  = masks['val_main']
val_storm_mask = masks['val_storm']

print(masks.keys())

In [ ]:
# ── Candidate Model Training ──────────────────────────────────────────────
# Train_1 + Train_2, h*=7h, MODEL_C features

X_train_full = feat_data.loc[train_mask, MODEL_C_OMNI_DNEUTRON]
y_train_full = feat_data.loc[train_mask, 'dst_target_7h']

In [ ]:
# XGBoost
xgb_pipe = Pipeline([('model', XGBoostDst())])
xgb_pipe.fit(X_train_full, y_train_full)

In [ ]:
# LightGBM
lgbm_pipe = Pipeline([('model', LightGBMDst())])
lgbm_pipe.fit(X_train_full, y_train_full)

### Reference Model Performance

Both models are first evaluated using the reference hyperparameter configuration. This establishes a baseline against which the tuned models can be compared. Since the forecasting horizon, feature set and training data remain fixed, any performance differences observed after tuning can be attributed to the optimisation of the model hyperparameters.

The better-performing reference model is not selected at this stage — model selection is performed only after hyperparameter optimisation and validation on both Val_Main and Val_Storm.

In [ ]:
# ── Evaluate on Val_Main and Val_Storm ───────────────────────────────────

EVAL_SEGMENTS_FULL = {
    'val_main' : val_main_mask,
    'val_storm': val_storm_mask,
}

# XGBoost
xgb_metrics_ref = {
    seg_name: run_segment(
        model_name      = 'xgb_ref',
        seg_name        = seg_name,
        models          = {7: xgb_pipe},
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_C_OMNI_DNEUTRON],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, 'dst_target_7h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = [7],
    )
    for seg_name, seg_mask in EVAL_SEGMENTS_FULL.items()
}

# LightGBM
lgbm_metrics_ref = {
    seg_name: run_segment(
        model_name      = 'lgbm_ref',
        seg_name        = seg_name,
        models          = {7: lgbm_pipe},
        X_seg_fn        = lambda h: feat_data.loc[seg_mask, MODEL_C_OMNI_DNEUTRON],
        y_true_fn       = lambda h: feat_data.loc[seg_mask, 'dst_target_7h'],
        y_train         = y_train,
        y_persist_fn    = lambda h: feat_data.loc[seg_mask, 'dst'].values,
        storm_thr       = STORM_THR,
        k_horizons      = [7],
    )
    for seg_name, seg_mask in EVAL_SEGMENTS_FULL.items()
}

print('\n── Reference performance (before tuning) ────────────────────────')
print(f'{"Model":<12} {"Segment":<12} {"RMSE":>8} {"StormRMSE":>12} {"MASE":>8}')
print('─' * 55)
for model_name, metrics in [('XGBoost', xgb_metrics_ref), ('LightGBM', lgbm_metrics_ref)]:
    for seg_name in EVAL_SEGMENTS_FULL:
        m = metrics[seg_name][7]
        print(f'{model_name:<12} {seg_name:<12} {m["rmse"]:>8.2f} {m["storm_rmse"]:>12.2f} {m["mase"]:>8.3f}')

In [ ]:
print(f"Scientific Validation MODEL_C val_main h=7h Storm RMSE: {model_c_metrics['val_main'][7]['storm_rmse']:.2f} nT")
print(f"Operational Forecasting XGBoost ref   val_main h=7h Storm RMSE: {xgb_metrics_ref['val_main'][7]['storm_rmse']:.2f} nT")
print(f"Operational Forecasting LightGBM ref  val_main h=7h Storm RMSE: {lgbm_metrics_ref['val_main'][7]['storm_rmse']:.2f} nT")

> **Observations — Reference Model Performance (h=7h):**
>
> Training on Train_1 + Train_2 provides a stronger operational training set than Train_1 alone, adding the declining phase of Solar Cycle 23 and increasing storm diversity. On Val_Main, XGBoost achieves Storm RMSE = 27.29 nT and LightGBM achieves 27.03 nT — both lower than the Train_1-only configuration from the scientific validation (MODEL_C Storm RMSE = 28.86 nT), suggesting that the additional training period improves storm generalisation.
>
> At the reference configuration, XGBoost and LightGBM perform almost identically. LightGBM gives marginally lower Storm RMSE on Val_Main (27.03 vs 27.29 nT), while XGBoost performs better on Val_Storm (102.37 vs 104.25 nT). The differences are small and should not be interpreted as a final model choice before hyperparameter optimisation.
>
> Val_Storm remains the limiting case for both models — Storm RMSE exceeds 100 nT for both, reflecting the extreme nature of the Halloween 2003 superstorm ($D_{st}$ = −422 nT) relative to the training distribution.
>
> These results establish the reference performance against which tuned models will be compared. Since the forecasting horizon, feature set and training data are fixed, any improvements after hyperparameter optimisation can be attributed to the optimisation process rather than changes in data selection or feature engineering.

### Hyperparameter Optimisation

With the reference performance established, we now search for the hyperparameter configuration that minimises Storm RMSE. Both models are optimised via RandomizedSearchCV with n_iter=50 and random_state=42 on Train_1 + Train_2 using TimeSeriesSplit (5 folds) — ensuring that the validation fold always follows the training fold chronologically and no future data leaks into the fit.

Val_Main and Val_Storm are not used during hyperparameter optimisation — they serve as independent external validation periods evaluated only after the best configuration is selected.

**Why TimeSeriesSplit and not standard KFold:** standard KFold shuffles the data randomly, allowing future observations to appear in the training fold. For time series this introduces look-ahead bias — the model appears to perform better than it actually would in deployment. TimeSeriesSplit preserves the temporal order:

$$\text{Fold } k: \text{Train} = [1, \ldots, n_k], \quad \text{Val} = [n_k+1, \ldots, n_{k+1}]$$

Each fold adds more training data — the model is always evaluated on data it has never seen.

**Scoring metric:** negative Storm RMSE averaged across the TimeSeriesSplit validation folds. RandomizedSearchCV maximises score, so higher = lower Storm RMSE = better model.

**Search space:**

| Parameter | XGBoost | LightGBM | Rationale |
|---|---|---|---|
| `n_estimators` | 300, 500, 800, 1000 | 300, 500, 800, 1000 | More trees vs faster training |
| `max_depth` | 3, 5, 7 | 3, 5, 7 | Shallow vs deep trees |
| `learning_rate` | 0.005, 0.01, 0.05 | 0.005, 0.01, 0.05 | Conservative vs aggressive |
| `subsample` | 0.7, 0.8, 1.0 | 0.7, 0.8, 1.0 | Row subsampling ratio |
| `colsample_bytree` | 0.7, 0.8, 1.0 | 0.7, 0.8, 1.0 | Feature subsampling ratio |
| `num_leaves` | — | 15, 31, 63 | LightGBM leaf-wise complexity |

XGBoost covers 4×3×3×3×3 = 324 combinations; LightGBM adds `num_leaves` for 972 combinations. RandomizedSearchCV samples 50 random combinations from each — reducing search time from ~3.5h (exhaustive) to ~45 minutes per model while covering a broader region of the parameter space than a small fixed grid.

Full search results, CV scores per combination and fitted best estimators are recorded in `notebooks/hyperparameter_optimisation.ipynb` and saved to `models/hp_opt_results.pkl`.

In [ ]:
import joblib
xgb_best_params = joblib.load('models/xgb_best_params.pkl')
lgbm_best_params = joblib.load('models/lgbm_best_params.pkl')
print('XGBoost best:', xgb_best_params)
print('LightGBM best:', lgbm_best_params)

> **Observations — Hyperparameter Optimisation**
>
> Both models were optimised via RandomizedSearchCV with n_iter=50 and random_state=42 on Train_1 + Train_2 using TimeSeriesSplit (5 folds). Val_Main and Val_Storm were not used during optimisation.
>
> **Best configurations:**
>
> | Parameter | XGBoost | LightGBM |
> |---|---|---|
> | `n_estimators` | 500 | 500 |
> | `max_depth` | 5 | 3 |
> | `learning_rate` | 0.005 | 0.01 |
> | `subsample` | 0.8 | 1.0 |
> | `colsample_bytree` | 0.7 | 1.0 |
> | `num_leaves` | — | 15 |
> | **CV Storm RMSE** | **33.75 nT** | **33.87 nT** |
>
> - Both models converge toward conservative configurations — low learning rate (0.005–0.01) and moderate tree depth (3–5). This is consistent with the storm weighting ($w \approx 21.3$) making the loss surface more complex and requiring slower, more careful learning.
> - XGBoost favours partial feature subsampling (`colsample_bytree=0.7`) while LightGBM prefers using all features and all rows (`subsample=1.0`, `colsample_bytree=1.0`). This reflects the different tree-growing strategies — leaf-wise growth in LightGBM achieves sufficient regularisation through depth and leaf constraints alone.
> - The CV scores (33.75 and 33.87 nT) are not directly comparable to the reference val_main Storm RMSE (~27 nT) — CV folds are subsets of Train_1+Train_2 which contain a different storm distribution than val_main.
> - XGBoost achieves marginally better CV score (33.75 vs 33.87 nT). Whether this advantage holds on Val_Main and Val_Storm is the subject of the next section.

In [ ]:
print('sw_speed in MODEL_C_OMNI_DNEUTRON:', 'sw_speed' in MODEL_C_OMNI_DNEUTRON)

## TODO — hyperparameter_optimisation.ipynb

- [ ] Cell 2: Merge `mlflow.set_tracking_uri` into Cell 1 (Imports & Load)
- [ ] Cell 18: Add markdown before MLflow XGBoost logging cell
- [ ] Cell 21: Add markdown before MLflow LightGBM logging cell
- [ ] Cell 24: Remove debug cell (`print(mlflow.get_tracking_uri())`)
- [ ] Cell 25: Remove empty cell
- [ ] Update XGBoost markdown (Cell 16) — search space table is outdated (shows old grid, not RandomizedSearchCV params)
- [ ] Update LightGBM markdown (Cell 19) — update n_iter comment to 50
- [ ] Add observations markdown after Cell 21 (LightGBM MLflow logging)
- [ ] Add final observations markdown comparing XGBoost vs LightGBM CV scores

### Model Validation

With the best hyperparameter configurations identified, both tuned models are now evaluated on Val_Main and Val_Storm — the independent external validation periods that were not used at any point during optimisation. This is the first honest comparison between the tuned models and the reference baseline.

**What we expect to see:** the tuned models should outperform the reference configuration on Val_Main Storm RMSE. Whether the improvement generalises to Val_Storm (Halloween 2003, $D_{st}$ = −422 nT) is the key question — if tuning helps on Val_Main but not Val_Storm, the model has overfitted to the routine storm regime.

**Design:** both models are loaded from `models/xgb_best_estimator.pkl` and `models/lgbm_best_estimator.pkl`. No retraining occurs here. Evaluation follows the same protocol as the scientific validation.

**Improvement metric:**

$$\Delta \text{StormRMSE} = \text{StormRMSE}_{\text{ref}} - \text{StormRMSE}_{\text{tuned}}$$

Positive $\Delta$ indicates improvement after tuning. Negative indicates regression.

**Evaluation metrics:**

**RMSE** — overall forecast accuracy:
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_t (y_t - \hat{y}_t)^2}$$
where $y_t$ is observed $D_{st}$, $\hat{y}_t$ is predicted $D_{st}$, $n$ is number of evaluation hours. Penalises large errors quadratically — a 20 nT error counts 4× more than a 10 nT error.

**Storm RMSE** — accuracy during geomagnetic storms (primary operational metric):
$$\text{StormRMSE} = \sqrt{\frac{1}{n_s}\sum_{t:\, y_t < -50} (y_t - \hat{y}_t)^2}$$
where $n_s$ is the number of storm hours ($y_t < -50$ nT). Measures how well the model performs when it matters most.

**MASE** (Hyndman & Koehler, 2006 [HYN06]) — scale-free accuracy relative to one-step persistence:
$$\text{MASE} = \frac{\text{MAE}(y, \hat{y})}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t^{\text{train}} - y_{t-1}^{\text{train}}|}$$
Numerator is mean absolute error on the evaluation set. Denominator is mean absolute one-step difference on Train_1 — a scale-free baseline that does not depend on the forecast horizon. MASE < 1 means the model outperforms naive one-step persistence.

**$R^2$** — proportion of variance explained (H_skill threshold):
$$R^2 = 1 - \frac{\sum_t (y_t - \hat{y}_t)^2}{\sum_t (y_t - \bar{y})^2}$$
where $\bar{y}$ is mean of observed $D_{st}$. $R^2 = 0$ means the model performs no better than predicting the mean; $R^2 = 1$ means perfect prediction. H_skill requires $R^2 \geq 0.60$ at $h^* = 7$h.

**DM statistic** (Diebold & Mariano, 1995 [DM95]) — statistical significance of improvement over persistence:
$$\text{DM} = \frac{\bar{d}}{\sqrt{\hat{V}(\bar{d})}}$$
where $\bar{d}$ is the mean loss differential between persistence and the candidate model, $\hat{V}(\bar{d})$ is the Newey-West HAC variance estimate corrected for $h$-step-ahead autocorrelation. A significantly negative DM statistic (p < 0.05, one-sided) indicates the candidate model is significantly more accurate than persistence.

**Peak timing error** — storm onset prediction accuracy:
$$\text{PTE} = \hat{t}_{\min} - t_{\min}$$
where $\hat{t}_{\min}$ is the hour of predicted $D_{st}$ minimum and $t_{\min}$ is the hour of observed minimum. Positive = model predicts storm peak too late; negative = too early. NaN if no storm present in the evaluation window.

In [ ]:
# ── Model Validation ──────────────────────────────────────────────────────
xgb_best  = joblib.load('models/xgb_best_estimator.pkl')
lgbm_best = joblib.load('models/lgbm_best_estimator.pkl')
TUNED_MODELS = {
    'XGBoost' : xgb_best,
    'LightGBM': lgbm_best,
}

def evaluate_tuned_model(model_name, model, seg_name, seg_mask):
    """Evaluate tuned model on one segment. Returns metrics dict."""
    y_true    = feat_data.loc[seg_mask, 'dst_target_7h']
    y_pred    = model.predict(feat_data.loc[seg_mask, MODEL_C_OMNI_DNEUTRON])
    y_persist = feat_data.loc[seg_mask, 'dst'].values
    m = compute_metrics(
        y_true    = y_true,
        y_pred    = y_pred,
        y_train   = y_train,
        y_persist = y_persist,
        storm_thr = STORM_THR,
        horizon   = 7,
    )
    print(f'  {seg_name:<12} | RMSE={m["rmse"]:6.2f} | '
          f'StormRMSE={m["storm_rmse"]:6.2f} | '
          f'MASE={m["mase"]:5.3f} | R²={m["r2"]:6.4f}')
    return m

def run_tuned_segment(model_name, model, seg_name, seg_mask):
    """Evaluate one segment and log nested MLflow child run."""
    m = evaluate_tuned_model(model_name, model, seg_name, seg_mask)
    with mlflow.start_run(run_name=f'{model_name.lower()}_tuned_{seg_name}', nested=True):
        mlflow.set_tag('model_type',     model_name.lower())
        mlflow.set_tag('evaluation_set', seg_name)
        mlflow.set_tag('tuned',          'true')
        mlflow.log_params({
            'horizon'    : 7,
            'storm_thr'  : STORM_THR,
            'feature_set': 'MODEL_C_OMNI_DNEUTRON',
        })
        mlflow.log_metrics({
            k: float(v) for k, v in m.items()
            if isinstance(v, (int, float)) and np.isfinite(float(v))
        })
    return m

def run_tuned_model(model_name, model):
    """Evaluate tuned model on all segments. Opens parent MLflow run."""
    print(f'\n── {model_name} (tuned) ──────────────────────────────────────')
    with mlflow.start_run(run_name=f'{model_name.lower()}_tuned'):
        mlflow.set_tag('model_type', model_name.lower())
        mlflow.set_tag('tuned',      'true')
        return {
            seg_name: run_tuned_segment(model_name, model, seg_name, seg_mask)
            for seg_name, seg_mask in EVAL_SEGMENTS.items()
        }

# ── Run ───────────────────────────────────────────────────────────────────
tuned_metrics = {
    model_name: run_tuned_model(model_name, model)
    for model_name, model in TUNED_MODELS.items()
}

joblib.dump(tuned_metrics, 'models/metrics_tuned_models.pkl')
print('\nSaved: models/metrics_tuned_models.pkl')

### Storm Weighting Strategy

Initial validation with inverse-frequency weighting ($w \approx 21.3$) revealed a systematic bias — both models converged to predicting exclusively negative $D_{st}$ values (`y_pred_max = -1.56 nT` on train). The combination of aggressive storm weighting and slow learning rate caused the model to overfit toward the storm regime.

The weighting strategy was revised to intensity-proportional weights:

$$w_t = \text{clip}\left(\frac{|D_{st}(t)|}{50},\, 1.0,\, 10.0\right) \quad \text{for } D_{st}(t) < -50 \text{ nT}$$

This gives weights proportional to storm intensity — $w = 2.0$ at $D_{st} = -100$ nT, $w = 4.0$ at $-200$ nT, and $w = 8.44$ at $-422$ nT (Halloween 2003, capped at 10.0) — preventing extreme events from dominating the loss while preserving storm sensitivity. The revised strategy is implemented in `XGBoostDstV2` and `LightGBMDstV2` and used for all subsequent training in this section.

In [ ]:
# ── Retrain with best params and V2 weighting ─────────────────────────────
xgb_best_params  = joblib.load('models/xgb_best_params.pkl')
lgbm_best_params = joblib.load('models/lgbm_best_params.pkl')

# Премахваме 'model__' prefix за директно подаване
def strip_prefix(params):
    return {k.replace('model__', ''): v for k, v in params.items()}

xgb_v2 = Pipeline([('model', XGBoostDstV2(**strip_prefix(xgb_best_params)))])
lgbm_v2 = Pipeline([('model', LightGBMDstV2(**strip_prefix(lgbm_best_params)))])

xgb_v2.fit(X_train_full, y_train_full)
lgbm_v2.fit(X_train_full, y_train_full)

# ── Verify predictions ────────────────────────────────────────────────────
y_pred_xgb = xgb_v2.predict(feat_data.loc[train_mask, MODEL_C_OMNI_DNEUTRON])
print(f'XGBoostV2 y_pred max: {y_pred_xgb.max():.2f}')
print(f'XGBoostV2 y_pred min: {y_pred_xgb.min():.2f}')

In [ ]:
TUNED_MODELS_V2 = {
    'XGBoostV2' : xgb_v2,
    'LightGBMV2': lgbm_v2,
}

tuned_metrics_v2 = {
    model_name: run_tuned_model(model_name, model)
    for model_name, model in TUNED_MODELS_V2.items()
}

joblib.dump(tuned_metrics_v2, 'models/metrics_tuned_models_v2.pkl')
print('\nSaved: models/metrics_tuned_models_v2.pkl')

> **Observations — Model Validation (V2 weighting, h=7h):**
>
> Switching to intensity-proportional storm weighting ($w = \text{clip}(|D_{st}|/50, 1.0, 10.0)$) resolves the prediction bias observed with V1 — both models now predict positive $D_{st}$ values during quiet periods (XGBoostV2 `y_pred_max = 6.24 nT`, LightGBMV2 `y_pred_max = 10.67 nT`).
>
> **R² improves substantially:** from negative values (V1: −0.095 and −0.068) to positive (V2: +0.497 and +0.481) on val_main. V1 had effectively learned to predict a storm-biased constant rather than tracking $D_{st}$ dynamics.
>
> **Storm RMSE deteriorates:** V2 Storm RMSE (32.29 and 31.70 nT) is worse than both V1 (21.51 and 22.50 nT) and the reference configuration (27.86 and 26.85 nT). The intensity-proportional weighting is more conservative than inverse-frequency weighting — storm hours receive lower weight on average ($w \leq 10$ vs $w \approx 21.3$), reducing storm sensitivity.
>
> **Val_storm remains difficult:** Storm RMSE ≈ 110 nT for both V2 models — slightly worse than V1 (101–106 nT) but both are comparable to persistence (102.29 nT). The Halloween 2003 superstorm remains outside the model's generalisation capacity at this stage.
>
> **XGBoostV2 and LightGBMV2 produce virtually identical results on val_main:**
>
> | Metric | XGBoostV2 | LightGBMV2 |
> |---|---|---|
> | RMSE | 11.05 nT | 11.22 nT |
> | Storm RMSE | 32.29 nT | 31.70 nT |
> | MASE | 2.616 | 2.620 |
> | $R^2$ | 0.4966 | 0.4814 |
>
> Differences are below 1 nT on all metrics. Two gradient boosting implementations with different tree-growing strategies have reached the same information ceiling under the same feature set and horizon. This is consistent with the result being robust to the choice of boosting algorithm rather than an artefact of a specific implementation.
>
> **Conclusion:** Neither V1 nor V2 weighting is optimal. V1 achieves better Storm RMSE but produces a biased model with negative R². V2 achieves positive R² but at the cost of Storm RMSE. A balanced weighting strategy — between $w \approx 21.3$ (V1) and $w \leq 10$ (V2) — is likely to perform better and is identified as the primary model improvement for the refinement stage. Note that the V2 models were evaluated with the hyperparameter configuration optimised for V1 weighting — rerunning RandomizedSearchCV with V2 weighting may recover some of the Storm RMSE loss.

### Residual Diagnostics

Residual diagnostics assess whether the model has captured all predictable structure in the data. If the residuals $e_t = D_{st}(t+7) - \hat{D}_{st}(t+7)$ behave as white noise — zero mean, constant variance, no autocorrelation — the model has extracted all available information and no further correction is warranted.

If significant autocorrelation is present, the residuals contain unexploited temporal structure that a secondary AR or SARIMA model could correct.

**ACF (Autocorrelation Function):** measures the correlation between $e_t$ and $e_{t-k}$ at lag $k$. Significant ACF at lag $k$ indicates that the residual at time $t$ is predictable from the residual $k$ hours earlier.

**PACF (Partial Autocorrelation Function):** measures the direct correlation between $e_t$ and $e_{t-k}$ after removing the effect of all intermediate lags $1, \ldots, k-1$. PACF is used to identify the order of an AR correction — significant PACF at lag $p$ with no significant PACF beyond $p$ indicates an AR($p$) structure.

**Decision rule:**
- No significant ACF/PACF beyond lag 1 → white noise → no residual correction
- Significant ACF/PACF at short lags → AR correction
- Significant ACF/PACF at seasonal lags (24h, 27×24h) → SARIMA correction

Both LightGBMV2 and XGBoostV2 residuals are analysed on Val_Main — the longer segment with more storm variety than Val_Storm.

In [ ]:
# ── Residual Diagnostics ──────────────────────────────────────────────────
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import acf, pacf

pio.renderers.default = 'notebook'

# ── Compute residuals on Val_Main ─────────────────────────────────────────
y_true_val     = feat_data.loc[val_main_mask, 'dst_target_7h'].dropna()
idx            = y_true_val.index
residuals_xgb  = y_true_val.values - xgb_v2.predict(feat_data.loc[idx, MODEL_C_OMNI_DNEUTRON])
residuals_lgbm = y_true_val.values - lgbm_v2.predict(feat_data.loc[idx, MODEL_C_OMNI_DNEUTRON])

print(f'XGBoostV2  residuals — mean: {residuals_xgb.mean():.3f} | std: {residuals_xgb.std():.3f}')
print(f'LightGBMV2 residuals — mean: {residuals_lgbm.mean():.3f} | std: {residuals_lgbm.std():.3f}')
print(f'Durbin-Watson XGBoostV2  : {durbin_watson(residuals_xgb):.4f}')
print(f'Durbin-Watson LightGBMV2 : {durbin_watson(residuals_lgbm):.4f}')

# ── Compute ACF/PACF ──────────────────────────────────────────────────────
lags = 48
acf_xgb,  confint_acf_xgb   = acf(residuals_xgb,  nlags=lags, alpha=0.05)
pacf_xgb, confint_pacf_xgb  = pacf(residuals_xgb, nlags=lags, alpha=0.05)
acf_lgbm, confint_acf_lgbm  = acf(residuals_lgbm, nlags=lags, alpha=0.05)
pacf_lgbm,confint_pacf_lgbm = pacf(residuals_lgbm,nlags=lags, alpha=0.05)

lag_x = list(range(lags + 1))

# ── Plot ──────────────────────────────────────────────────────────────────
fig = make_subplots(rows=2, cols=2,
    subplot_titles=[
        'ACF — XGBoostV2 residuals',
        'PACF — XGBoostV2 residuals',
        'ACF — LightGBMV2 residuals',
        'PACF — LightGBMV2 residuals',
    ])

for row, (acf_vals, pacf_vals, conf_a, conf_p) in enumerate([
    (acf_xgb,  pacf_xgb,  confint_acf_xgb,  confint_pacf_xgb),
    (acf_lgbm, pacf_lgbm, confint_acf_lgbm, confint_pacf_lgbm),
], start=1):
    for col, (vals, conf) in enumerate([
        (acf_vals,  conf_a),
        (pacf_vals, conf_p),
    ], start=1):
        fig.add_trace(go.Bar(
            x=lag_x, y=vals,
            marker_color='steelblue',
            showlegend=False,
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=lag_x, y=conf[:, 1] - vals,
            mode='lines', line=dict(color='red', dash='dash'),
            name='95% CI', showlegend=(row==1 and col==1),
        ), row=row, col=col)
        fig.add_trace(go.Scatter(
            x=lag_x, y=conf[:, 0] - vals,
            mode='lines', line=dict(color='red', dash='dash'),
            showlegend=False,
        ), row=row, col=col)

fig.update_xaxes(title_text='Lag (hours)')
fig.update_yaxes(title_text='Correlation')
fig.update_layout(
    title='Residual Diagnostics — ACF/PACF (Val_Main, h=7h)',
    height=700,
)
fig.write_image('images/fig_residual_diagnostics.png', scale=2)
fig.show()

In [ ]:
# ── Hybrid XGBoost–AR(2) with 7-hour delayed residual feedback ───────────
from statsmodels.tsa.ar_model import AutoReg

# ── Step 1: Fit XGBoostV2 on Train (NaN-safe) ────────────────────────────
train_idx  = feat_data.index[train_mask & feat['dst_target_7h'].notna()]
X_train    = feat_data.loc[train_idx, MODEL_C_OMNI_DNEUTRON]
y_train_h7 = feat_data.loc[train_idx, 'dst_target_7h']

xgb_v2.fit(X_train, y_train_h7)

# ── Step 2: Compute train residuals ──────────────────────────────────────
residuals_train = y_train_h7.values - xgb_v2.predict(X_train)

# ── Step 3: Fit AutoReg(2) on train residuals ─────────────────────────────
ar_model = AutoReg(residuals_train, lags=2).fit()
phi1 = ar_model.params[1]
phi2 = ar_model.params[2]
c    = ar_model.params[0]
print(f'AR(2) on train residuals: c={c:.4f}, φ1={phi1:.4f}, φ2={phi2:.4f}')

# ── Step 4: Build residual history from train ─────────────────────────────
residual_by_time = {}
for ts, res in zip(feat_data.loc[train_idx, 'datetime'], residuals_train):
    residual_by_time[pd.Timestamp(ts)] = res

# ── Step 5: Walk-forward on Val_Main with 7-hour delayed feedback ─────────
y_true_val = feat_data.loc[val_main_mask, 'dst_target_7h'].dropna()
idx        = y_true_val.index
xgb_preds  = xgb_v2.predict(feat_data.loc[idx, MODEL_C_OMNI_DNEUTRON])
timestamps = feat_data.loc[idx, 'datetime'].apply(pd.Timestamp)

final_preds             = []
missing_feedback_points = 0

for ts, xgb_pred, true_y in zip(timestamps, xgb_preds, y_true_val.values):
    ts7 = ts - pd.Timedelta(hours=7)
    ts8 = ts - pd.Timedelta(hours=8)

    if ts7 not in residual_by_time or ts8 not in residual_by_time:
        missing_feedback_points += 1

    e_t7 = residual_by_time.get(ts7, 0.0)
    e_t8 = residual_by_time.get(ts8, 0.0)

    ar_correction = c + phi1 * e_t7 + phi2 * e_t8
    final_preds.append(xgb_pred + ar_correction)

    # Update residual history after observing true value
    residual_by_time[ts] = true_y - xgb_pred

print(f'Missing delayed residual feedback points: {missing_feedback_points}')
final_preds = np.array(final_preds)

# ── Step 6: Evaluate ──────────────────────────────────────────────────────
y_persist_val = feat_data.loc[idx, 'dst'].values

m_xgb = compute_metrics(
    y_true    = y_true_val.values,
    y_pred    = xgb_preds,
    y_train   = y_train_h7.values,
    y_persist = y_persist_val,
    storm_thr = STORM_THR,
    horizon   = 7,
)

m_hybrid = compute_metrics(
    y_true    = y_true_val.values,
    y_pred    = final_preds,
    y_train   = y_train_h7.values,
    y_persist = y_persist_val,
    storm_thr = STORM_THR,
    horizon   = 7,
)

print(f'\n{"Model":<35} {"RMSE":>8} {"StormRMSE":>12} {"R²":>8}')
print('─' * 67)
print(f'{"XGBoostV2":<35} {m_xgb["rmse"]:>8.2f} {m_xgb["storm_rmse"]:>12.2f} {m_xgb["r2"]:>8.4f}')
print(f'{"XGBoostV2 + AR(2) delayed":<35} {m_hybrid["rmse"]:>8.2f} {m_hybrid["storm_rmse"]:>12.2f} {m_hybrid["r2"]:>8.4f}')

joblib.dump({'xgb': m_xgb, 'hybrid': m_hybrid}, 'models/metrics_hybrid_val_main.pkl')
print('\nSaved: models/metrics_hybrid_val_main.pkl')

print(
    f'Missing delayed residual feedback points: '
    f'{missing_feedback_points}/{len(idx)} '
    f'({100 * missing_feedback_points / len(idx):.2f}%)'
)

> **Observations — Hybrid XGBoost–AR(2) with 7-hour Delayed Residual Feedback**
>
> The hybrid model was trained exclusively on residuals from Train_1 + Train_2 and evaluated on Val_Main using a walk-forward procedure with a 7-hour delayed residual feedback. No information from Val_Main was used during model fitting. Only 8 of 52,542 validation samples (0.02%) lacked delayed residual feedback, indicating that the evaluation closely represents the intended operational setting.
>
> The AR(2) model fitted on the training residuals produced the coefficients $c = 0.124$, $\phi_1 = 1.023$, and $\phi_2 = -0.143$. The coefficient $\phi_1 \approx 1$ suggests strong persistence in the training residuals, indicating that consecutive prediction errors exhibit substantial temporal dependence.
>
> | Model | RMSE | Storm RMSE | $R^2$ |
> |---|---|---|---|
> | XGBoostV2 | 11.05 nT | 32.29 nT | 0.4966 |
> | XGBoostV2 + delayed AR(2) | 11.55 nT | 39.24 nT | 0.4507 |
>
> Despite the strong autocorrelation observed in the training residuals, the delayed AR(2) correction did not improve predictive performance on Val_Main. Instead, the Storm RMSE increased by 6.95 nT and the coefficient of determination decreased, indicating that the residual dynamics learned during training did not generalise to the validation period.
>
> A plausible explanation is that the statistical properties of the residual process differ between the training and validation intervals, which correspond to different phases of the solar cycle and exhibit different storm frequencies. However, this hypothesis was not tested directly and therefore cannot be considered a confirmed cause of the observed degradation.
>
> **Conclusion.** Residual diagnostics demonstrate strong autocorrelation in the prediction errors (Durbin–Watson ≈ 0.21, $\rho_1 \approx 0.90$). However, the autoregressive model estimated from the training residuals did not improve performance on the validation period. This suggests that the residual dynamics are not sufficiently stable for a static AR(2) correction to generalise across the train–validation split. The presence of autocorrelation in the residuals is therefore a necessary but not sufficient condition for a static AR residual model to improve out-of-sample forecasting performance.
>
> Consequently, **XGBoostV2** is selected as the final forecasting model, while the hybrid AR(2) approach is retained as a diagnostic experiment and a potential direction for future work — for example, using out-of-fold residuals for AR fitting or allowing the AR coefficients to adapt across solar cycle phases.

## References  

[BUR75]  Burton, R. K., McPherron, R. L., & Russell, C. T. (1975). An empirical relationship between interplanetary conditions and Dst. Journal of Geophysical Research, 80(31), 4204–4214. DOI:10.1029/JA080i031p04204

[KIN05]  King, J. H., & Papitashvili, N. E. (2005). Solar wind spatial scales in and comparisons of hourly Wind and ACE plasma and magnetic field data. Journal of Geophysical Research, 110, A02209. DOI:10.1029/2004JA010649

[PAP20]  Papitashvili, N. E., & King, J. H. (2020). OMNI 1-min Data. NASA Space Physics Data Facility. DOI:10.48322/1SHR-HT18

[KIS25]  Kisvárdai, I., et al. (2025). Analysis of 42 Years of Cosmic Ray Measurements by the Neutron Monitor at Lomnický štít. Earth and Space Science. DOI:10.1029/2024EA003656

[NAI23]  Nair, M., et al. (2023). MagNet — A Data-Science Competition to Predict Disturbance Storm-Time Index (Dst) From Solar Wind Data. Space Weather, 21, e2023SW003514. DOI:10.1029/2023SW003514

[HU23]   Hu, Q., et al. (2023). Multi-Hour Ahead Dst Index Prediction Using Multi-Fidelity Boosted Neural Networks. Space Weather. DOI:10.1029/2022SW003286

[ABD22]  Abduallah, Y., et al. (2022). DeepSPaCE: Deep Learning for Solar Proton and CME Events. arXiv:2205.02447

[WDC]    World Data Center for Geomagnetism, Kyoto. Dst Index. https://wdc.kugi.kyoto-u.ac.jp

[SCI22]  Collado-Villaverde, A., et al. (2022). Prominence of the training data preparation in geomagnetic storm prediction using deep neural networks. Scientific Reports, 12, 9095. DOI:10.1038/s41598-022-11721-8

[ROD10]  Rodriguez, J. V., et al. (2010). The GOES Solar Proton Events: Comparison with IMP-8 and ACE. Space Weather. DOI:10.1029/2009SW000556

[CAM19]  Camporeale, E. (2019). The Challenge of Machine Learning in Space Weather: Nowcasting and Forecasting. Space Weather, 17, 1166–1207. DOI:10.1029/2018SW002061

[BEL00]  Belov, A. V. (2000). Large Scale Modulation: View from the Earth. Space Science Reviews, 93, 79–105. DOI:10.1023/A:1026538008532

[KE17] Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. NeurIPS 2017.

[DM95]   Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. Journal of Business and Economic Statistics, 13(3), 253–263. DOI:10.1080/07350015.1995.10524599

[HYN06]  Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. International Journal of Forecasting, 22(4), 679–688. DOI:10.1016/j.ijforecast.2006.03.001

[LAP20]  Laperre, B., Amaya, J., & Lapenta, G. (2020). Dynamic Time Warping as a New Evaluation for Dst Forecast With Machine Learning. Frontiers in Astronomy and Space Sciences, 7, 39. DOI:10.3389/fspas.2020.00039

[ISH09]  Ishkov, V. N. (2009). Characteristics of solar activity in the extended minimum phase of solar cycles 23–24. In V. N. Obridko & Yu. A. Nagovitsyn (Eds.), Activity Cycles on the Sun and Stars (pp. 57). St. Petersburg: VVM.

[DM95]   Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. Journal of Business and Economic Statistics, 13(3), 253–263. DOI:10.1080/07350015.1995.10524599

[HYN06]  Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of forecast accuracy. International Journal of Forecasting, 22(4), 679–688. DOI:10.1016/j.ijforecast.2006.03.001

[LAP20]  Laperre, B., Amaya, J., & Lapenta, G. (2020). Dynamic Time Warping as a New Evaluation for Dst Forecast With Machine Learning. Frontiers in Astronomy and Space Sciences, 7, 39. DOI:10.3389/fspas.2020.00039

[ROD14]  Rodriguez, J. V., Krosschell, J. C., & Green, J. C. (2014). Intercalibration of GOES 8–15 solar proton detectors. Space Weather, 12, 92–109. DOI:10.1002/2013SW000996

[CHE16]  Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining, 785–794. DOI:10.1145/2939672.2939785

[KE17]   Ke, G., Meng, Q., Finley, T., Wang, T., Chen, W., Ma, W., Ye, Q., & Liu, T.-Y. (2017). LightGBM: A highly efficient gradient boosting decision tree. Advances in Neural Information Processing Systems, 30, 3146–3154.